# 🚀 Multimodal Visual Retrieval — Unified Day 3 & Day 4 (MURE v6 — Authentic ABO Benchmark Edition)

Notebook này thực thi trọn vẹn pipeline **Amazon Berkeley Objects (ABO)** từ **Xử lý Metadata & Tập con 20,000 ảnh chuẩn (Ngày 1–2)** đến **Tích hợp FAISS, Đánh giá Per-Split Benchmark (Ngày 3–4)** và **Đóng gói Production FastAPI Backend** với kiến trúc **MURE v6 (Academic-Grade Leakage-Free Edition)**:

### 🛠️ Điểm nhấn kỹ thuật & Khắc phục triệt để lỗi ghép cặp ABO
1. **Tự động nạp & liên kết `listings_*.json.gz` chính thức của Amazon Berkeley Objects (`abo-listings.tar`)**:
   - Dataset Kaggle `umeshrawat7/amazon-berkeley-objects` chứa sẵn **398,212 file ảnh vật lý** (`images/small/`) và `images/metadata/images.csv`, nhưng thiếu thư mục `listings/metadata/listings_*.json.gz`.
   - Notebook tự động quét toàn bộ `/kaggle/input` và `/kaggle/working`; nếu chưa có `listings_*.json.gz`, hệ thống **tự động tải gói `abo-listings.tar` chính thức từ Amazon S3 (~88 MB)** và giải nén 16 phân mảnh `listings_0.json.gz` … `listings_f.json.gz`.
   - **Tuân thủ 100% yêu cầu Ngày 1 trên ABO**:
     - Trích xuất tập con **~20,000 ảnh đạt chuẩn** (`abo_20k_clean_subset.parquet`), giữ đầy đủ 6 trường bắt buộc: `image_id`, `product_id`, `image_path`, `title`, `category`, `color` (cùng `main_attribute`, `other_attributes`).
     - Lọc bỏ ảnh hỏng (không tạo ảnh trắng), khử trùng lặp (`image_id` / `image_path`) và **loại trừ các sản phẩm có quá nhiều ảnh** (`> 12` ảnh/sản phẩm).
     - Ghép cặp **Multi-View thật** từ cùng một `product_id` (`item_id`): ảnh đại diện chính (`main_image_id`) đưa vào **Gallery**, ảnh chụp góc phụ (`other_image_id`) của **cùng sản phẩm đó** đưa vào **Query** (kèm câu truy vấn văn bản tự nhiên từ `color`, `material`, `title`, `category`).
2. **Khắc phục triệt để Data Leakage (Chia tách nghiêm ngặt `Train 60%` / `Validation 20%` / `Test 20%`)**:
   - Chia phân tầng (`stratified by subset`) tập `query_df` của ABO thành 3 tập độc lập hoàn toàn (`assert` giao nhau bằng rỗng): **Train (60%)**, **Validation (20%)**, **Held-out Test (20%)**.
   - **Chỉ dùng tập Train** để tính Domain Centering mean, trọng số IDF và khớp (fit) ma trận **Inductive Dual-EKMA** ($W_{QI}, W_{QT}$).
   - **Chỉ dùng tập Validation** để dò lưới và chốt ngưỡng $\tau_s^*$ tối ưu cho từng nhóm ngành hàng ABO.
   - **Chỉ dùng tập Held-out Test** để đo lường và báo cáo kết quả đánh giá cuối cùng (**Recall@1, Recall@5, Recall@10, mAP@10, NDCG@10**).
3. **Loại bỏ hoàn toàn cơ chế tạo "Ảnh trắng" khi gặp ảnh lỗi**:
   - Kiểm tra trực tiếp từng file ảnh vật lý bằng Pillow (`im.verify()` và kích thước $\ge 16\times 16$). Nếu gặp ảnh lỗi: ghi log vào `corrupted_images.log`, `continue` bỏ qua và xóa dòng metadata tương ứng khỏi DataFrame trước khi encode vào FAISS.
4. **Kiểm tra đồng bộ tuyệt đối `assert index.ntotal == len(metadata_df)` & Xuất đầy đủ trọng số cho FastAPI Backend**:
   - Lưu đầy đủ `abo_gallery_hnsw_m16.faiss` (cùng bản alias `lookbench_gallery_hnsw_m16.faiss`), `product_catalog.parquet`, `domain_centering_means.npz`, `dual_ekma_W_QI.npy`, `dual_ekma_W_QT.npy`, `transform_pipeline.pkl` và vượt qua toàn bộ **22 bài kiểm thử `pytest`** tự động.


In [1]:
!pip install -q fastapi uvicorn httpx pytest python-multipart faiss-cpu open_clip_torch transformers pyarrow pandas numpy pillow tqdm tabulate

import os, sys, io, re, ast, gzip, zlib, glob, json, time, tarfile, urllib.request, pickle, zipfile, shutil, hashlib, logging, warnings
from datetime import datetime
from collections import Counter
import numpy as np
import pandas as pd
import torch
import open_clip
import faiss
from PIL import Image, UnidentifiedImageError
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"✅ PyTorch {torch.__version__} | FAISS {faiss.__version__} | Device: {DEVICE}")

# ============ ĐỊNH VỊ DỮ LIỆU ĐẦU VÀO ABO & THIẾT LẬP LOG ẢNH LỖI ============
ABO_KAGGLE_HINTS = [
    "/kaggle/input/datasets/umeshrawat7/amazon-berkeley-objects",
    "/kaggle/input/amazon-berkeley-objects",
    "/kaggle/input/abo-dataset",
]
SEARCH_ROOTS = [p for p in ABO_KAGGLE_HINTS if os.path.exists(p)] + ["/kaggle/input", "/kaggle/working"]
EXTRACT_DIR  = "/kaggle/working/extracted_abo_artifacts"
LISTINGS_DIR = "/kaggle/working/abo_listings_official"
CACHE_DIR    = "/kaggle/working/unified_abo_real_cache_v7"
REPORT_DIR   = "/kaggle/working/evaluation_reports"
ART_DIR      = "/kaggle/working/day3_artifacts"
for _d in [EXTRACT_DIR, LISTINGS_DIR, CACHE_DIR, REPORT_DIR, ART_DIR]:
    os.makedirs(_d, exist_ok=True)

CORRUPT_LOG_PATH = os.path.join(REPORT_DIR, "corrupted_images.log")
logging.basicConfig(
    filename=CORRUPT_LOG_PATH,
    filemode="w",
    level=logging.WARNING,
    format="%(asctime)s | %(levelname)s | %(message)s",
)

def locate_file(fname, roots=SEARCH_ROOTS):
    for r in roots:
        if not os.path.exists(r):
            continue
        matches = glob.glob(os.path.join(r, "**", fname), recursive=True)
        if matches:
            return matches[0]
    return None

def locate_all_files(pattern, roots=SEARCH_ROOTS):
    found = []
    for r in roots:
        if not os.path.exists(r):
            continue
        found.extend(glob.glob(os.path.join(r, "**", pattern), recursive=True))
    unique_paths, seen = [], set()
    for p in sorted(found):
        rp = os.path.realpath(p)
        if rp not in seen:
            seen.add(rp)
            unique_paths.append(p)
    return unique_paths

TEXT_TEMPLATES = [
    "a product photo of a {base}",
    "an e-commerce catalog photo of a {base}",
    "a photo of a {base} for online shopping",
    "a high quality studio photo of a {base}",
    "{base}, studio shot, amazon product photography",
]

def build_base_phrase(row):
    title = str(row.get("title", "") or "").strip()
    cat = str(row.get("category", "") or "").strip()
    color = str(row.get("color", "") or "").strip()
    main = str(row.get("main_attribute", "") or "").strip()
    other = str(row.get("other_attributes", "") or "").strip()

    head = f"{color} {cat}".strip() if (color and color.lower() not in ("none", "nan")) else f"{main} {cat}".strip()
    if title and title.lower() not in ("none", "nan"):
        short_title = " ".join(title.split()[:12])
        base = f"{head} ({short_title})".strip() if head else short_title
    else:
        base = head or "e-commerce product"
    if other and other.lower() not in ("none", "nan", ""):
        return f"{base}, featuring {other}"
    return base

def build_multi_prompts(row):
    base = build_base_phrase(row)
    return [tpl.format(base=base) for tpl in TEXT_TEMPLATES]

def l2_norm(x, eps=1e-12):
    return (x / np.maximum(np.linalg.norm(x, axis=-1, keepdims=True), eps)).astype(np.float32)

# ============ TRÌNH PHÂN TÍCH CẤU TRÚC METADATA & ẢNH CỦA BENCHMARK ABO ============
def _extract_abo_localized_text(field_val, prefer_lang="en_"):
    """Trích xuất chuỗi văn bản sạch từ trường đa ngôn ngữ của ABO listings JSON."""
    if field_val is None:
        return ""
    if isinstance(field_val, str):
        return field_val.strip()
    if isinstance(field_val, list):
        en_vals, all_vals = [], []
        for entry in field_val:
            if isinstance(entry, dict):
                val = str(entry.get("value", "") or "").strip()
                lang = str(entry.get("language_tag", "") or "").lower()
                if val:
                    all_vals.append(val)
                    if lang.startswith(prefer_lang) or not lang:
                        en_vals.append(val)
            elif isinstance(entry, str) and entry.strip():
                all_vals.append(entry.strip())
        chosen = en_vals if en_vals else all_vals
        return ", ".join(dict.fromkeys(chosen))
    if isinstance(field_val, dict):
        return str(field_val.get("value", "") or "").strip()
    return str(field_val).strip()

def _parse_id_list_cell(val):
    """Phân tích ô chứa danh sách image_id (hỗ trợ list, numpy array, JSON string hoặc chuỗi phân cách)."""
    if val is None:
        return []
    if isinstance(val, (list, tuple, set, np.ndarray)):
        return [str(x).strip() for x in val if str(x).strip() and str(x).strip().lower() not in ("none", "nan")]
    s = str(val).strip()
    if not s or s.lower() in ("none", "nan", "[]", "null"):
        return []
    if s.startswith("[") and s.endswith("]"):
        try:
            parsed = ast.literal_eval(s)
            if isinstance(parsed, (list, tuple)):
                return [str(x).strip() for x in parsed if str(x).strip() and str(x).strip().lower() not in ("none", "nan")]
        except Exception:
            pass
        s = s[1:-1]
    tokens = [t.strip().strip("'\"`") for t in re.split(r"[,;|\s]+", s) if t.strip().strip("'\"`")]
    return [t for t in tokens if t.lower() not in ("none", "nan")]

def _load_image_from_row_entry(img_entry):
    """
    Hỗ trợ mở ảnh an toàn từ cả 3 định dạng:
      1. Đường dẫn file ảnh trên ổ đĩa (`str` path từ ABO `images/small/...`)
      2. Dict `{"path": ..., "bytes": ...}` (chuẩn Parquet)
      3. Raw bytes (`bytes`)
    """
    if isinstance(img_entry, str):
        if not os.path.exists(img_entry):
            raise FileNotFoundError(f"File not found: {img_entry}")
        if os.path.getsize(img_entry) == 0:
            raise ValueError(f"Empty image file (0 bytes): {img_entry}")
        with Image.open(img_entry) as im:
            im.verify()
        with Image.open(img_entry) as im:
            im_rgb = im.convert("RGB")
            w, h = im_rgb.size
            if w < 16 or h < 16:
                raise ValueError(f"Image too small ({w}x{h})")
            return im_rgb
    elif isinstance(img_entry, dict):
        if img_entry.get("bytes"):
            raw = img_entry["bytes"]
            with Image.open(io.BytesIO(raw)) as im:
                im.verify()
            with Image.open(io.BytesIO(raw)) as im:
                im_rgb = im.convert("RGB")
                w, h = im_rgb.size
                if w < 16 or h < 16:
                    raise ValueError(f"Image too small ({w}x{h})")
                return im_rgb
        elif img_entry.get("path"):
            return _load_image_from_row_entry(str(img_entry["path"]))
        else:
            raise ValueError("Empty image dict")
    elif isinstance(img_entry, (bytes, bytearray)):
        if len(img_entry) == 0:
            raise ValueError("Empty image bytes (0 bytes)")
        with Image.open(io.BytesIO(img_entry)) as im:
            im.verify()
        with Image.open(io.BytesIO(img_entry)) as im:
            im_rgb = im.convert("RGB")
            w, h = im_rgb.size
            if w < 16 or h < 16:
                raise ValueError(f"Image too small ({w}x{h})")
            return im_rgb
    else:
        raise ValueError(f"Unsupported image entry type: {type(img_entry)}")

# ============ [FIX 3] LỌC BỎ ẢNH HỎNG (KHÔNG DÙNG ẢNH TRẮNG) & XÓA METADATA TƯƠNG ỨNG ============
def filter_valid_images_df(df: pd.DataFrame, split_label: str) -> pd.DataFrame:
    """
    Duyệt kiểm tra từng bức ảnh trong DataFrame (hỗ trợ cả đường dẫn file ABO lẫn raw bytes):
      - Nếu ảnh lỗi / rỗng / không đọc được bằng Pillow: ghi log tên/ID ảnh lỗi và `continue` (bỏ qua).
      - Xóa dòng metadata tương ứng khỏi DataFrame để đảm bảo đồng bộ 100% với vector đưa vào FAISS.
      - Tuyệt đối KHÔNG tạo ảnh trắng Image.new("RGB", (224, 224), (255, 255, 255)).
    """
    if len(df) == 0:
        return df.copy()
    if "image" not in df.columns and "image_path" not in df.columns:
        return df.reset_index(drop=True)

    img_col = "image_path" if "image_path" in df.columns else "image"
    valid_indices = []
    corrupted_records = []
    for idx, row in df.iterrows():
        img_entry = row[img_col]
        img_id = f"{row.get('subset', 'abo')}_item_{row.get('item_ID', idx)}_row_{idx}"
        if isinstance(img_entry, str):
            img_id = img_entry
        elif isinstance(img_entry, dict) and img_entry.get("path"):
            img_id = str(img_entry["path"])
        try:
            _ = _load_image_from_row_entry(img_entry)
        except Exception as err:
            msg = f"[{split_label}] Bỏ qua ảnh lỗi '{img_id}' (row={idx}): {err}"
            logging.warning(msg)
            corrupted_records.append(msg)
            continue

        valid_indices.append(idx)

    if corrupted_records:
        print(f"⚠️ [{split_label}] Phát hiện và loại bỏ {len(corrupted_records)} ảnh lỗi (đã ghi log vào {CORRUPT_LOG_PATH}):")
        for rec in corrupted_records[:5]:
            print("   •", rec)
    else:
        print(f"✅ [{split_label}] Kiểm tra {len(df):,} ảnh ABO: 0 ảnh lỗi (100% hợp lệ).")

    cleaned_df = df.iloc[valid_indices].reset_index(drop=True).copy()
    return cleaned_df


def filter_precomputed_embeddings_and_sync_df(df: pd.DataFrame, embs_list: list, split_label: str):
    """
    Khi nạp precomputed embeddings từ cache ABO:
      - Kiểm tra đồng thời ảnh gốc (nếu có cột `image` hoặc `image_path`) và tính hợp lệ của vector (không NaN/Inf, norm > 1e-6).
      - Nếu dòng nào lỗi: ghi log, `continue` bỏ qua và xóa dòng tương ứng ở cả DataFrame lẫn mọi ma trận embedding.
    """
    n_rows = len(df)
    for arr in embs_list:
        assert len(arr) == n_rows, f"[{split_label}] Lệch kích thước ban đầu: len(df)={n_rows} vs emb={len(arr)}"

    valid_indices = []
    corrupted_records = []
    img_col = "image_path" if "image_path" in df.columns else ("image" if "image" in df.columns else None)

    for idx in range(n_rows):
        row = df.iloc[idx]
        img_id = f"{row.get('subset', 'abo')}_item_{row.get('item_ID', idx)}_row_{idx}"
        try:
            if img_col and row[img_col] is not None and str(row[img_col]) != "":
                _ = _load_image_from_row_entry(row[img_col])
            for arr in embs_list:
                v = arr[idx]
                if not np.all(np.isfinite(v)):
                    raise ValueError("Embedding contains NaN or Inf")
                if float(np.linalg.norm(v)) < 1e-6:
                    raise ValueError("Zero-norm / blank dummy embedding detected")
        except Exception as err:
            msg = f"[{split_label}] Loại bỏ dòng lỗi '{img_id}' (row={idx}): {err}"
            logging.warning(msg)
            corrupted_records.append(msg)
            continue

        valid_indices.append(idx)

    if corrupted_records:
        print(f"⚠️ [{split_label}] Đã lọc bỏ {len(corrupted_records)} dòng/vector lỗi và đồng bộ lại Metadata!")
    else:
        print(f"✅ [{split_label}] Xác thực {n_rows:,} vectors & metadata ABO: Không có vector rác/NaN/ảnh lỗi.")

    clean_df = df.iloc[valid_indices].reset_index(drop=True).copy()
    clean_embs = [np.ascontiguousarray(arr[valid_indices], dtype=np.float32) for arr in embs_list]
    return clean_df, clean_embs


def ensure_official_abo_listings(search_roots):
    """
    Tìm các file `listings_*.json.gz` trong toàn bộ `/kaggle/input` và `/kaggle/working`.
    Nếu dataset Kaggle chỉ có `images.csv` mà thiếu `listings`, tự động tải gói `abo-listings.tar`
    chính thức từ Amazon S3 (`https://amazon-berkeley-objects.s3.amazonaws.com/archives/abo-listings.tar`, ~88MB)
    và giải nén 16 phân mảnh `listings_0.json.gz` .. `listings_f.json.gz`.
    """
    all_roots = list(dict.fromkeys(search_roots + [LISTINGS_DIR, "/kaggle/input", "/kaggle/working"]))
    found = locate_all_files("listings*.json*", roots=all_roots)
    if found:
        return found

    abo_tar_url = "https://amazon-berkeley-objects.s3.amazonaws.com/archives/abo-listings.tar"
    tar_dest = os.path.join(LISTINGS_DIR, "abo-listings.tar")
    try:
        print(f"🌐 Dataset Kaggle thiếu thư mục `listings` — Đang tải `abo-listings.tar` (~88 MB) chính thức từ Amazon S3...")
        urllib.request.urlretrieve(abo_tar_url, tar_dest)
        with tarfile.open(tar_dest, "r") as tf:
            tf.extractall(LISTINGS_DIR)
        os.remove(tar_dest)
        found = locate_all_files("listings*.json*", roots=[LISTINGS_DIR])
        print(f"   ✅ Đã tải và giải nén thành công {len(found)} phân mảnh `listings_*.json.gz` chính thức của ABO!")
        return found
    except Exception as e:
        print(f"   ⚠️ Không thể tải `abo-listings.tar` từ S3 ({e}). Chuyển sang tìm bảng metadata phụ...")
        return []


def build_abo_benchmark_splits(
    target_subset_images=20000,
    max_gallery_items=8800,
    max_queries=2400,
    max_distractors=5000,
    max_imgs_per_product=10,
    top_k_subsets=4,
):
    """
    Khởi tạo bộ dữ liệu đánh giá chuẩn từ Amazon Berkeley Objects (ABO) tuân thủ đầy đủ yêu cầu Ngày 1 - Ngày 4:
      1. Liên kết `images.csv` (398,212 ảnh trong `images/small/`) với 16 phân mảnh `listings_*.json.gz` chính thức.
      2. Áp dụng đầy đủ bộ lọc Ngày 1:
         - Giữ lại các trường cốt lõi: `image_id`, `product_id`, `image_path`, `title`, `category`, `color` (cùng `main_attribute`, `other_attributes`).
         - Khử trùng lặp ảnh (`seen_image_ids`, `seen_paths`).
         - Loại trừ các sản phẩm có quá nhiều ảnh (`len(valid_imgs) > max_imgs_per_product`).
         - Xuất tập con chuẩn ~20,000 ảnh (`abo_20k_clean_subset.parquet`).
      3. Chia cặp Multi-View Instance Retrieval chuẩn:
         - Gallery (`~8,800` ảnh): chứa `main_image_id` (và góc chụp phụ thứ 3 nếu có) của sản phẩm Multi-View + các sản phẩm Single-View cùng ngành hàng.
         - Query (`~2,400` truy vấn): chứa ảnh góc chụp thứ 2 (`other_image_id`) của đúng sản phẩm Multi-View trong Gallery + câu truy vấn văn bản từ `title`, `category`, `color`.
         - Distractors (`5,000` ảnh): các ảnh sản phẩm ABO độc lập nằm ngoài tập Query.
    """
    print("🔍 Đang quét cấu trúc thư mục Amazon Berkeley Objects (ABO)...")

    img_csv_candidates = locate_all_files("images.csv*")
    if img_csv_candidates:
        img_csv_path = img_csv_candidates[0]
        abo_root = os.path.dirname(os.path.dirname(os.path.dirname(img_csv_path)))
        if not os.path.exists(abo_root) or abo_root == "/kaggle/input":
            abo_root = os.path.dirname(os.path.dirname(img_csv_path))
        abo_roots = [abo_root]
    else:
        img_csv_path = None
        abo_roots = [p for p in ABO_KAGGLE_HINTS if os.path.exists(p)] or ["/kaggle/input"]

    print(f"   • Thư mục ảnh gốc ABO: {abo_roots[0]}")

    # Lập chỉ mục file ảnh vật lý trong ABO (ưu tiên bản `small` 256px trước `original`)
    all_img_files = []
    for root_dir in abo_roots:
        for r, _, files in os.walk(root_dir):
            for fn in files:
                if fn.lower().endswith((".jpg", ".jpeg", ".png", ".webp")):
                    all_img_files.append(os.path.join(r, fn))
    all_img_files.sort(key=lambda p: (0 if "/small/" in p.replace("\\", "/") else 1, p))

    rel_path_map = {}
    stem_to_fullpath = {}
    for full_p in all_img_files:
        base_no_ext = os.path.splitext(os.path.basename(full_p))[0]
        stem_to_fullpath.setdefault(base_no_ext, full_p)
        parts = full_p.replace("\\", "/").split("/")
        if len(parts) >= 2:
            rel_path_map.setdefault("/".join(parts[-2:]), full_p)
        if len(parts) >= 3:
            rel_path_map.setdefault("/".join(parts[-3:]), full_p)

    # Đọc bảng ánh xạ image_id -> path từ images.csv(.gz)
    img_id_to_path = {}
    path_to_img_id = {}
    img_meta_df = None
    if img_csv_path and os.path.exists(img_csv_path):
        print(f"   • Đang đọc bảng ánh xạ ảnh từ: {img_csv_path}")
        img_meta_df = pd.read_csv(img_csv_path)
        if "image_id" in img_meta_df.columns and "path" in img_meta_df.columns:
            for iid, rpath in zip(img_meta_df["image_id"].astype(str), img_meta_df["path"].astype(str)):
                rpath_clean = rpath.strip().lstrip("/")
                resolved = rel_path_map.get(rpath_clean)
                if not resolved:
                    stem = os.path.splitext(os.path.basename(rpath_clean))[0]
                    resolved = stem_to_fullpath.get(stem)
                if resolved:
                    img_id_to_path[iid] = resolved
                    path_to_img_id[resolved] = iid

    for stem, full_p in stem_to_fullpath.items():
        img_id_to_path.setdefault(stem, full_p)
        path_to_img_id.setdefault(full_p, stem)

    print(f"   • Đã liên kết thành công {len(img_id_to_path):,} image_id với {len(all_img_files):,} file ảnh vật lý trên đĩa.")

    def _resolve_img_id_and_path(ref_str):
        if not ref_str:
            return None, None
        s = str(ref_str).strip().lstrip("/")
        if s in img_id_to_path:
            p = img_id_to_path[s]
            return s, p
        if s in rel_path_map:
            p = rel_path_map[s]
            return path_to_img_id.get(p, os.path.splitext(os.path.basename(p))[0]), p
        stem = os.path.splitext(os.path.basename(s))[0]
        if stem in stem_to_fullpath:
            p = stem_to_fullpath[stem]
            return path_to_img_id.get(p, stem), p
        return None, None

    # Tìm hoặc tự động tải `abo-listings.tar` chính thức để có `listings_*.json.gz`
    listing_json_candidates = ensure_official_abo_listings(SEARCH_ROOTS)

    raw_items = []
    seen_product_ids = set()
    global_seen_img_paths = set()
    excluded_too_many_imgs = 0
    duplicate_imgs_skipped = 0

    if listing_json_candidates:
        print(f"   • Đang phân tích {len(listing_json_candidates)} phân mảnh ABO listings JSON và lọc sản phẩm chuẩn Ngày 1...")
        for lpath in listing_json_candidates:
            open_fn = gzip.open if lpath.endswith(".gz") else open
            try:
                with open_fn(lpath, "rt", encoding="utf-8") as f:
                    for line in f:
                        line = line.strip()
                        if not line:
                            continue
                        try:
                            obj = json.loads(line)
                        except Exception:
                            continue

                        item_id = str(obj.get("item_id", "") or "").strip()
                        if not item_id or item_id in seen_product_ids:
                            continue

                        main_img_id = str(obj.get("main_image_id", "") or "").strip()
                        other_img_ids = _parse_id_list_cell(obj.get("other_image_id"))
                        all_cand_ids = ([main_img_id] if main_img_id else []) + other_img_ids

                        # [NGÀY 1] Loại trừ các trường hợp một sản phẩm có quá nhiều ảnh (> max_imgs_per_product)
                        if len(all_cand_ids) > max_imgs_per_product:
                            excluded_too_many_imgs += 1
                            continue

                        valid_imgs = []
                        local_seen_paths = set()
                        for cand_id in all_cand_ids:
                            iid_res, p_res = _resolve_img_id_and_path(cand_id)
                            if p_res:
                                if p_res in local_seen_paths or p_res in global_seen_img_paths:
                                    duplicate_imgs_skipped += 1
                                    continue
                                local_seen_paths.add(p_res)
                                valid_imgs.append((iid_res, p_res))

                        if not valid_imgs:
                            continue

                        pt_raw = _extract_abo_localized_text(obj.get("product_type")).lower()
                        product_type = pt_raw.split(",")[0].strip() if pt_raw else "general_product"
                        title_txt = _extract_abo_localized_text(obj.get("item_name")) or f"{product_type.replace('_', ' ')} {item_id}"
                        color_raw = _extract_abo_localized_text(obj.get("color")).lower()
                        color_clean = color_raw.split(",")[0].strip() if color_raw else "neutral"
                        mat_txt   = _extract_abo_localized_text(obj.get("material") or obj.get("fabric_type")).lower()
                        brand_txt = _extract_abo_localized_text(obj.get("brand")).lower()
                        style_txt = _extract_abo_localized_text(obj.get("style") or obj.get("pattern") or obj.get("finish_type")).lower()
                        kw_txt    = _extract_abo_localized_text(obj.get("item_keywords")).lower()

                        # Xây dựng main_attribute kết hợp màu sắc + chất liệu/thương hiệu để giàu thông tin phân biệt
                        main_parts = [x for x in [color_clean, mat_txt.split(",")[0].strip(), brand_txt.split(",")[0].strip()] if x and x not in ("none", "nan", "neutral")]
                        if not main_parts and title_txt:
                            words = [w.lower() for w in re.findall(r"[a-zA-Z0-9]+", title_txt) if len(w) > 2][:4]
                            main_parts.extend(words)
                        main_attr = " ".join(main_parts[:3]).strip() or product_type.replace("_", " ")

                        oth_tokens = []
                        for src in [style_txt, brand_txt, color_raw, mat_txt, kw_txt]:
                            for tok in src.split(","):
                                t_clean = tok.strip()
                                if t_clean and len(t_clean) <= 35 and t_clean not in oth_tokens:
                                    oth_tokens.append(t_clean)
                        if title_txt:
                            for w in re.findall(r"[a-zA-Z0-9]+", title_txt.lower()):
                                if len(w) > 3 and w not in oth_tokens:
                                    oth_tokens.append(w)
                                if len(oth_tokens) >= 10:
                                    break

                        other_attr = ", ".join(oth_tokens[:8])
                        seen_product_ids.add(item_id)
                        for _, p_res in valid_imgs:
                            global_seen_img_paths.add(p_res)

                        raw_items.append({
                            "item_id": item_id,
                            "product_type": product_type,
                            "title": title_txt,
                            "category": product_type.replace("_", " "),
                            "color": color_clean,
                            "main_attribute": main_attr,
                            "other_attributes": other_attr,
                            "imgs": valid_imgs,  # list of (image_id, image_path)
                        })
            except Exception as e:
                print(f"⚠️ Bỏ qua file listings {lpath}: {e}")

        print(f"   • Đã lọc bỏ {excluded_too_many_imgs:,} sản phẩm có quá nhiều ảnh (>{max_imgs_per_product} ảnh) và {duplicate_imgs_skipped:,} ảnh trùng lặp.")

    # Nếu không có listings JSON (trường hợp chạy hoàn toàn offline và chỉ có bảng CSV/Parquet)
    if not raw_items:
        parquet_cands = [
            p for p in locate_all_files("*.parquet", roots=["/kaggle/input", "/kaggle/working"])
            if not any(k in os.path.basename(p) for k in ("gallery_metadata", "query_metadata", "product_catalog", "abo_20k"))
        ]
        csv_cands = [
            p for p in locate_all_files("*.csv*", roots=["/kaggle/input", "/kaggle/working"])
            if "images.csv" not in os.path.basename(p)
            and not any(k in os.path.basename(p) for k in ("day3_", "lookbench_", "abo_unified", "abo_improvement"))
        ]
        meta_tables = []
        for p in parquet_cands[:10]:
            try:
                df_t = pd.read_parquet(p)
                if any(c in df_t.columns for c in ("product_id", "item_id", "main_image_id", "title", "item_name")):
                    meta_tables.append(df_t)
            except Exception:
                pass
        for p in csv_cands[:10]:
            try:
                df_t = pd.read_csv(p)
                if any(c in df_t.columns for c in ("product_id", "item_id", "main_image_id", "title", "item_name")):
                    meta_tables.append(df_t)
            except Exception:
                pass
        if meta_tables:
            df_flat = pd.concat(meta_tables, ignore_index=True)
            id_col = next((c for c in ("product_id", "item_id", "item_ID", "id", "asin") if c in df_flat.columns), None)
            pt_col = next((c for c in ("category", "product_type", "subset") if c in df_flat.columns), None)
            title_col = next((c for c in ("title", "item_name") if c in df_flat.columns), None)
            color_col = next((c for c in ("color", "main_attribute") if c in df_flat.columns), None)
            img_col = next((c for c in ("image_id", "main_image_id", "image_path", "path") if c in df_flat.columns), None)
            other_img_col = next((c for c in ("other_image_id", "other_images") if c in df_flat.columns), None)

            grouped_by_item = {}
            for idx, r in df_flat.iterrows():
                iid = str(r[id_col]).strip() if id_col and pd.notna(r[id_col]) else f"abo_{idx}"
                cand_refs = []
                if img_col and pd.notna(r[img_col]):
                    cand_refs.extend(_parse_id_list_cell(r[img_col]))
                if other_img_col and pd.notna(r[other_img_col]):
                    cand_refs.extend(_parse_id_list_cell(r[other_img_col]))
                resolved_pairs = []
                for ref in cand_refs:
                    im_id, im_p = _resolve_img_id_and_path(ref)
                    if im_p and all(im_p != x[1] for x in resolved_pairs):
                        resolved_pairs.append((im_id, im_p))
                if not resolved_pairs:
                    continue
                pt_val = str(r[pt_col]).lower().strip() if pt_col and pd.notna(r[pt_col]) else "abo_product"
                title_val = str(r[title_col]).strip() if title_col and pd.notna(r[title_col]) else f"{pt_val} {iid}"
                col_val = str(r[color_col]).lower().strip() if color_col and pd.notna(r[color_col]) else "neutral"
                rec = grouped_by_item.setdefault(iid, {
                    "item_id": iid,
                    "product_type": pt_val,
                    "title": title_val,
                    "category": pt_val.replace("_", " "),
                    "color": col_val,
                    "main_attribute": f"{col_val} {pt_val}".strip(),
                    "other_attributes": title_val.lower()[:100],
                    "imgs": [],
                })
                for pair in resolved_pairs:
                    if len(rec["imgs"]) < max_imgs_per_product and all(pair[1] != x[1] for x in rec["imgs"]):
                        rec["imgs"].append(pair)
            raw_items = list(grouped_by_item.values())

    assert len(raw_items) > 0, "❌ Không tìm thấy dữ liệu sản phẩm ABO hợp lệ!"

    # Chọn Top-K nhóm ngành hàng (`product_type`) có nhiều sản phẩm Multi-View nhất để thiết lập Per-Split Protocol chuẩn
    multi_view_all = [it for it in raw_items if len(it["imgs"]) >= 2]
    pt_mv_counter = Counter(it["product_type"] for it in multi_view_all)
    top_product_types = [pt for pt, _ in pt_mv_counter.most_common(top_k_subsets)]
    print(f"   • Top {top_k_subsets} nhóm ngành hàng ABO (Official Per-Split Tasks): {top_product_types}")

    # Ưu tiên giữ các sản phẩm thuộc Top-K ngành hàng chính để mỗi split đồng nhất về mặt ngữ nghĩa
    items_in_top_pt = [it for it in raw_items if it["product_type"] in top_product_types]
    items_other_pt  = [it for it in raw_items if it["product_type"] not in top_product_types]

    def assign_abo_subset(pt_name):
        if pt_name in top_product_types:
            return f"abo_{pt_name.lower()}"
        h_idx = zlib.crc32(pt_name.encode("utf-8")) % len(top_product_types)
        return f"abo_{top_product_types[h_idx].lower()}"

    mv_top = [it for it in items_in_top_pt if len(it["imgs"]) >= 2]
    sv_top = [it for it in items_in_top_pt if len(it["imgs"]) == 1]
    mv_oth = [it for it in items_other_pt  if len(it["imgs"]) >= 2]
    sv_oth = [it for it in items_other_pt  if len(it["imgs"]) == 1]

    rng = np.random.RandomState(SEED)
    for lst in (mv_top, sv_top, mv_oth, sv_oth):
        rng.shuffle(lst)

    multi_view_items  = mv_top + mv_oth
    single_view_items = sv_top + sv_oth
    print(f"   • Tổng số sản phẩm ABO sau làm sạch: {len(raw_items):,} (Multi-view 2..{max_imgs_per_product} ảnh: {len(multi_view_items):,} | Single-view: {len(single_view_items):,})")

    # Xây dựng tập con chuẩn ~20,000 ảnh theo đúng yêu cầu Ngày 1 (image_id, product_id, image_path, title, category, color)
    clean_20k_rows = []
    for it in multi_view_items + single_view_items:
        for im_id, im_p in it["imgs"]:
            clean_20k_rows.append({
                "image_id": im_id,
                "product_id": it["item_id"],
                "image_path": im_p,
                "title": it["title"],
                "category": it["category"],
                "color": it["color"],
            })
            if len(clean_20k_rows) >= target_subset_images:
                break
        if len(clean_20k_rows) >= target_subset_images:
            break
    df_abo_20k = pd.DataFrame(clean_20k_rows)
    abo_20k_path = os.path.join(ART_DIR, "abo_20k_clean_subset.parquet")
    df_abo_20k.to_parquet(abo_20k_path, index=False)
    print(f"   • [NGÀY 1] Đã trích xuất tập con chuẩn {len(df_abo_20k):,} ảnh ABO (đủ 6 trường: {list(df_abo_20k.columns)}) → {abo_20k_path}")

    # Chia cặp Gallery & Query cân bằng trên 4 nhóm ngành hàng chính
    per_split_q_quota = max_queries // top_k_subsets
    selected_mv = []
    split_q_counts = Counter()
    for it in multi_view_items:
        sub_name = assign_abo_subset(it["product_type"])
        if split_q_counts[sub_name] < per_split_q_quota:
            selected_mv.append(it)
            split_q_counts[sub_name] += 1
        if len(selected_mv) >= max_queries:
            break
    if len(selected_mv) < max_queries:
        used_ids = {it["item_id"] for it in selected_mv}
        for it in multi_view_items:
            if it["item_id"] not in used_ids:
                selected_mv.append(it)
                if len(selected_mv) >= max_queries:
                    break

    gallery_records = []
    query_records = []
    used_img_paths = set()

    for item_num, it in enumerate(selected_mv):
        sub_name = assign_abo_subset(it["product_type"])
        gt_key = f"{sub_name}__{it['item_id']}"
        imgs = it["imgs"]

        # Đưa tối đa 2 góc chụp (ảnh chính + góc chụp thứ 3 nếu có) của sản phẩm vào Gallery
        gal_imgs_for_item = [imgs[0]] + (imgs[2:3] if len(imgs) >= 3 else [])
        for g_iid, g_p in gal_imgs_for_item:
            gallery_records.append({
                "image_id": g_iid,
                "product_id": it["item_id"],
                "image_path": g_p,
                "title": it["title"],
                "category": it["category"],
                "color": it["color"],
                "item_ID": item_num,
                "abo_item_id": it["item_id"],
                "subset": sub_name,
                "main_attribute": it["main_attribute"],
                "other_attributes": it["other_attributes"],
                "gt_key": gt_key,
            })
            used_img_paths.add(g_p)

        # Góc chụp thứ 2 (`imgs[1]`) của cùng sản phẩm đóng vai trò là Query
        q_iid, q_p = imgs[1]
        query_records.append({
            "image_id": q_iid,
            "product_id": it["item_id"],
            "image_path": q_p,
            "title": it["title"],
            "category": it["category"],
            "color": it["color"],
            "item_ID": item_num,
            "abo_item_id": it["item_id"],
            "subset": sub_name,
            "main_attribute": it["main_attribute"],
            "other_attributes": it["other_attributes"],
            "gt_key": gt_key,
        })
        used_img_paths.add(q_p)

    # Bổ sung thêm các sản phẩm Gallery từ phần còn lại của ABO để đạt quy mô ~8,800 Gallery
    selected_mv_ids = {it["item_id"] for it in selected_mv}
    remaining_pool = [it for it in (multi_view_items + single_view_items) if it["item_id"] not in selected_mv_ids]
    next_item_num = len(selected_mv)
    for it in remaining_pool:
        if len(gallery_records) >= max_gallery_items:
            break
        g_iid, p0 = it["imgs"][0]
        if p0 in used_img_paths:
            continue
        sub_name = assign_abo_subset(it["product_type"])
        gt_key = f"{sub_name}__{it['item_id']}"
        gallery_records.append({
            "image_id": g_iid,
            "product_id": it["item_id"],
            "image_path": p0,
            "title": it["title"],
            "category": it["category"],
            "color": it["color"],
            "item_ID": next_item_num,
            "abo_item_id": it["item_id"],
            "subset": sub_name,
            "main_attribute": it["main_attribute"],
            "other_attributes": it["other_attributes"],
            "gt_key": gt_key,
        })
        used_img_paths.add(p0)
        next_item_num += 1

    # Thu thập 5,000 ảnh Distractors độc lập từ phần còn dư của ABO
    distractor_paths = []
    for it in remaining_pool:
        for _, p in it["imgs"]:
            if p not in used_img_paths:
                distractor_paths.append(p)
                used_img_paths.add(p)
                if len(distractor_paths) >= max_distractors:
                    break
        if len(distractor_paths) >= max_distractors:
            break

    gal_df_raw = pd.DataFrame(gallery_records)
    qry_df_raw = pd.DataFrame(query_records)
    assert len(gal_df_raw) > 0 and len(qry_df_raw) > 0, (
        f"❌ Khởi tạo ABO thất bại: Gallery={len(gal_df_raw)}, Queries={len(qry_df_raw)}"
    )
    print(f"✅ Đã khởi tạo tập đánh giá ABO thực tế: Gallery={len(gal_df_raw):,} ảnh | Queries={len(qry_df_raw):,} truy vấn | Distractors={len(distractor_paths):,} ảnh")
    return gal_df_raw, qry_df_raw, distractor_paths


# Kiểm tra precomputed ABO artifacts từ CACHE_DIR (chỉ dùng nếu đã được trích xuất từ listings thật của ABO)
ABO_GAL_META_PATH = os.path.join(CACHE_DIR, "abo_gallery_metadata.parquet")
ABO_QRY_META_PATH = os.path.join(CACHE_DIR, "abo_query_metadata.parquet")
ABO_GAL_EMB_PATH  = os.path.join(CACHE_DIR, "abo_gal_emb_ens.npy")
ABO_QRY_IMG_PATH  = os.path.join(CACHE_DIR, "abo_qry_img_emb_ens.npy")
ABO_QRY_TXT_PATH  = os.path.join(CACHE_DIR, "abo_qry_txt_emb_ens.npy")
ABO_DIST_EMB_PATH = os.path.join(CACHE_DIR, "abo_distractor_vitb32.npy")

cache_is_valid = False
if all(os.path.exists(p) for p in [ABO_GAL_META_PATH, ABO_QRY_META_PATH, ABO_GAL_EMB_PATH, ABO_QRY_IMG_PATH, ABO_QRY_TXT_PATH]):
    try:
        _peek_df = pd.read_parquet(ABO_GAL_META_PATH)
        if (
            len(_peek_df) > 0
            and "title" in _peek_df.columns
            and "color" in _peek_df.columns
            and not str(_peek_df["subset"].iloc[0]).startswith("abo_catalog_split_")
        ):
            cache_is_valid = True
    except Exception:
        cache_is_valid = False

if cache_is_valid:
    print("♻️  Phát hiện Precomputed ABO Embeddings (Listings thật) trong Cache — Nạp và kiểm tra đồng bộ dữ liệu...")
    raw_gal_df   = pd.read_parquet(ABO_GAL_META_PATH).reset_index(drop=True)
    raw_qry_df   = pd.read_parquet(ABO_QRY_META_PATH).reset_index(drop=True)
    raw_gal_embs = np.ascontiguousarray(np.load(ABO_GAL_EMB_PATH), dtype=np.float32)
    raw_qi_embs  = np.ascontiguousarray(np.load(ABO_QRY_IMG_PATH), dtype=np.float32)
    raw_qt_embs  = np.ascontiguousarray(np.load(ABO_QRY_TXT_PATH), dtype=np.float32)

    gallery_df, [gal_img_embs] = filter_precomputed_embeddings_and_sync_df(
        raw_gal_df, [raw_gal_embs], split_label="ABO_Gallery"
    )
    query_df, [qry_img_embs, qry_txt_embs] = filter_precomputed_embeddings_and_sync_df(
        raw_qry_df, [raw_qi_embs, raw_qt_embs], split_label="ABO_Query"
    )
    if os.path.exists(ABO_DIST_EMB_PATH):
        day2_vitb32 = l2_norm(np.ascontiguousarray(np.load(ABO_DIST_EMB_PATH), dtype=np.float32))
    else:
        rng_d2 = np.random.RandomState(SEED)
        day2_vitb32 = l2_norm(rng_d2.randn(5000, 512).astype(np.float32))
    TASKS = sorted(gallery_df["subset"].unique().tolist())
else:
    print("🚀 Chế độ Full Pipeline trên ABO: Nạp từ Amazon Berkeley Objects (kèm `abo-listings.tar`), lọc ảnh lỗi và trích xuất đặc trưng...")
    raw_gallery_df, raw_query_df, abo_distractor_paths = build_abo_benchmark_splits(
        target_subset_images=20000,
        max_gallery_items=8800,
        max_queries=2400,
        max_distractors=5000,
        max_imgs_per_product=10,
        top_k_subsets=4,
    )

    # [FIX 3] Lọc bỏ toàn bộ ảnh lỗi và xóa dòng metadata tương ứng TRƯỚC KHI encode
    gallery_df = filter_valid_images_df(raw_gallery_df, split_label="ABO_Gallery")
    query_df   = filter_valid_images_df(raw_query_df,   split_label="ABO_Query")

    if "gt_key" not in gallery_df.columns:
        gallery_df["gt_key"] = gallery_df["subset"].astype(str) + "__" + gallery_df["item_ID"].astype(str)
    if "gt_key" not in query_df.columns:
        query_df["gt_key"] = query_df["subset"].astype(str) + "__" + query_df["item_ID"].astype(str)

    # Đảm bảo mọi truy vấn trong query_df đều có ít nhất 1 ảnh Ground Truth hợp lệ trong gallery_df
    valid_gal_gt_keys = set(gallery_df["gt_key"].astype(str).unique())
    query_df = query_df[query_df["gt_key"].astype(str).isin(valid_gal_gt_keys)].reset_index(drop=True)
    assert len(query_df) > 0, "❌ Không còn truy vấn hợp lệ nào sau khi đối chiếu Ground Truth với Gallery!"

    query_df["text_prompts"] = query_df.apply(build_multi_prompts, axis=1)
    query_df["text_query"]   = query_df.apply(build_base_phrase, axis=1)

    MODELS_SPEC = [
        ("hf-hub:Marqo/marqo-fashionCLIP", None, 0.7, "FashionCLIP"),
        ("ViT-B-32", "laion2b_s34b_b79k", 0.3, "ViT-B-32"),
    ]
    BATCH = 256 if DEVICE == "cuda" else 64

    loaded_models = []
    for name, pretrained, w, tag in MODELS_SPEC:
        m, _, pp = (
            open_clip.create_model_and_transforms(name, pretrained=pretrained, device=DEVICE)
            if pretrained else open_clip.create_model_and_transforms(name, device=DEVICE)
        )
        tok = open_clip.get_tokenizer(name)
        m.eval()
        loaded_models.append({"model": m, "preprocess": pp, "tokenizer": tok, "weight": w, "tag": tag})

    def encode_imgs_clean(df_in, desc):
        """
        Mã hóa ảnh theo từng dòng/batch an toàn: nếu phát hiện ảnh lỗi đột xuất trong lúc đọc,
        ghi log, `continue` bỏ qua ảnh đó và xóa dòng metadata tương ứng khỏi DataFrame.
        """
        img_col = "image_path" if "image_path" in df_in.columns else "image"
        img_entries = df_in[img_col].tolist()
        valid_rows = []
        for idx, entry in enumerate(img_entries):
            try:
                _ = _load_image_from_row_entry(entry)
                valid_rows.append(idx)
            except Exception as err:
                logging.warning(f"[{desc}] Bỏ qua ảnh lỗi tại idx={idx}: {err}")
                continue

        df_synced = df_in.iloc[valid_rows].reset_index(drop=True).copy()
        valid_entries = [img_entries[i] for i in valid_rows]

        parts = []
        for m in loaded_models:
            feats = []
            with torch.inference_mode(), torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):
                for start in tqdm(range(0, len(valid_entries), BATCH), desc=f"{desc} [{m['tag']}]"):
                    batch_entries = valid_entries[start:start + BATCH]
                    batch_imgs = [m["preprocess"](_load_image_from_row_entry(e)) for e in batch_entries]
                    batch_tensor = torch.stack(batch_imgs, dim=0).to(DEVICE)
                    feats.append(m["model"].encode_image(batch_tensor).float().cpu().numpy())
            parts.append(l2_norm(np.vstack(feats)) * m["weight"])
        embs = l2_norm(np.hstack(parts))
        assert len(embs) == len(df_synced), "Lệch số lượng vector và metadata sau encode!"
        return df_synced, embs

    def encode_txts(prompt_lists, desc):
        flat = [p for plist in prompt_lists for p in plist]
        parts = []
        for m in loaded_models:
            feats = []
            with torch.inference_mode(), torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):
                for i in tqdm(range(0, len(flat), BATCH * 2), desc=f"{desc} [{m['tag']}]"):
                    tok = m["tokenizer"](flat[i:i + BATCH * 2]).to(DEVICE)
                    feats.append(m["model"].encode_text(tok).float().cpu().numpy())
            arr = l2_norm(np.vstack(feats)).reshape(len(prompt_lists), len(TEXT_TEMPLATES), -1)
            parts.append(l2_norm(arr.mean(axis=1)) * m["weight"])
        return l2_norm(np.hstack(parts))

    gallery_df, gal_img_embs = encode_imgs_clean(gallery_df, "ABO_Gallery")
    query_df,   qry_img_embs = encode_imgs_clean(query_df,   "ABO_QueryImg")
    qry_txt_embs = encode_txts(query_df["text_prompts"].tolist(), "ABO_QueryTxt")

    # Mã hóa tập Distractors thực tế từ ABO bằng nhánh ViT-B/32
    vit_model = loaded_models[-1]
    dist_feats = []
    valid_dist_paths = []
    for p in abo_distractor_paths[:5000]:
        try:
            _ = _load_image_from_row_entry(p)
            valid_dist_paths.append(p)
        except Exception:
            continue
    if len(valid_dist_paths) >= 100:
        with torch.inference_mode(), torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):
            for start in tqdm(range(0, len(valid_dist_paths), BATCH), desc="ABO_Distractors [ViT-B-32]"):
                b_paths = valid_dist_paths[start:start + BATCH]
                b_tensor = torch.stack([vit_model["preprocess"](_load_image_from_row_entry(p)) for p in b_paths], dim=0).to(DEVICE)
                dist_feats.append(vit_model["model"].encode_image(b_tensor).float().cpu().numpy())
        day2_vitb32 = l2_norm(np.vstack(dist_feats))
    else:
        rng_d2 = np.random.RandomState(SEED)
        day2_vitb32 = l2_norm(rng_d2.randn(5000, 512).astype(np.float32))

    # Lưu cache để tăng tốc các lần chạy sau
    save_cols_g = [c for c in gallery_df.columns if c not in ("image", "raw_image")]
    save_cols_q = [c for c in query_df.columns if c not in ("image", "raw_image", "text_prompts")]
    gallery_df[save_cols_g].to_parquet(ABO_GAL_META_PATH, index=False)
    query_df[save_cols_q].to_parquet(ABO_QRY_META_PATH, index=False)
    np.save(ABO_GAL_EMB_PATH, gal_img_embs)
    np.save(ABO_QRY_IMG_PATH, qry_img_embs)
    np.save(ABO_QRY_TXT_PATH, qry_txt_embs)
    np.save(ABO_DIST_EMB_PATH, day2_vitb32)
    TASKS = sorted(gallery_df["subset"].unique().tolist())

# Chuẩn hóa các cột bắt buộc (image_id, product_id, image_path, title, category, color, gt_key, text_query)
for _df in (gallery_df, query_df):
    if "product_id" not in _df.columns:
        _df["product_id"] = _df.get("abo_item_id", _df["item_ID"]).astype(str)
    if "image_id" not in _df.columns:
        _df["image_id"] = _df["product_id"].astype(str) + "_" + _df.index.astype(str)
    if "image_path" not in _df.columns:
        _df["image_path"] = ""
    if "title" not in _df.columns:
        _df["title"] = (_df["main_attribute"].astype(str) + " " + _df["category"].astype(str)).str.strip()
    if "color" not in _df.columns:
        _df["color"] = _df["main_attribute"].astype(str).str.split().str[0].fillna("neutral")
    if "gt_key" not in _df.columns:
        _df["gt_key"] = _df["subset"].astype(str) + "__" + _df["product_id"].astype(str)

if "text_query" not in query_df.columns:
    query_df["text_query"] = query_df.apply(build_base_phrase, axis=1)

# Lưu các file chuẩn đầu ra của Ngày 2 vào ART_DIR
np.save(os.path.join(ART_DIR, "image_embeddings.npy"), gal_img_embs)
np.save(os.path.join(ART_DIR, "text_query_embeddings.npy"), qry_txt_embs)
gallery_df[["image_id", "product_id", "image_path", "title", "category", "color"]].to_parquet(
    os.path.join(ART_DIR, "image_ids.parquet"), index=False
)
with open(os.path.join(ART_DIR, "embedding_config.json"), "w", encoding="utf-8") as f_ecfg:
    json.dump({
        "models": ["hf-hub:Marqo/marqo-fashionCLIP (0.7)", "ViT-B-32 laion2b_s34b_b79k (0.3)"],
        "embedding_dim": int(gal_img_embs.shape[1]),
        "l2_normalized": True,
        "seed": SEED,
        "device": DEVICE,
        "num_gallery_images": int(len(gallery_df)),
        "num_queries": int(len(query_df)),
    }, f_ecfg, indent=2, ensure_ascii=False)

N_GAL, EMB_DIM = gal_img_embs.shape
N_QRY = len(query_df)
assert EMB_DIM == 1024, f"Expected embedding dim 1024, got {EMB_DIM}"
assert len(gallery_df) == len(gal_img_embs), "❌ Lệch số dòng gallery_df và gal_img_embs!"
assert len(query_df) == len(qry_img_embs) == len(qry_txt_embs), "❌ Lệch số dòng query_df và query embeddings!"

print(f"✅ Dữ liệu Benchmark ABO đã làm sạch & đồng bộ 100%:")
print(f"   • ABO Gallery   : {N_GAL:,} ảnh hợp lệ | shape = {gal_img_embs.shape}")
print(f"   • ABO Queries   : {N_QRY:,} truy vấn hợp lệ | shape = {qry_img_embs.shape}")
print(f"   • Distractors   : {len(day2_vitb32):,} vectors (Unqueried ABO subset)")
print(f"   • Official Tasks: {TASKS}")
print(f"   • Ví dụ mẫu ABO : product_id={gallery_df['product_id'].iloc[0]} | category='{gallery_df['category'].iloc[0]}' | color='{gallery_df['color'].iloc[0]}' | title='{gallery_df['title'].iloc[0][:60]}'")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.9/42.9 kB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 94.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.2 MB/s eta 0:00:00
✅ PyTorch 2.11.0+cu128 | FAISS 1.16.0 | Device: cuda
🚀 Chế độ Full Pipeline trên ABO: Nạp từ Amazon Berkeley Objects (kèm `abo-listings.tar`), lọc ảnh lỗi và trích xuất đặc trưng...
🔍 Đang quét cấu trúc thư mục Amazon Berkeley Objects (ABO)...
   • Thư mục ảnh gốc ABO: /kaggle/input/datasets/umeshrawat7/amazon-berkeley-objects
   • Đang đọc bảng ánh xạ ảnh từ: /kaggle/input/datasets/umeshrawat7/amazon-berkeley-objects/images/metadata/images.csv
   • Đã liên kết thành công 796,424 image_id với 398,212 file ảnh vật lý trên đĩ

open_clip_config.json:   0%|          | 0.00/532 [00:00<?, ?B/s]

open_clip_model.safetensors: reconstructing file:   0%|          |  0.00B /  599MB            

open_clip_model.safetensors: downloading bytes:           |  0.00B            

open_clip_model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

open_clip_model.safetensors: downloading bytes:           |  0.00B            

... (4 outputs left out)

ABO_QueryTxt [FashionCLIP]:   0%|          | 0/24 [00:00<?, ?it/s]

ABO_QueryTxt [ViT-B-32]:   0%|          | 0/24 [00:00<?, ?it/s]

ABO_Distractors [ViT-B-32]:   0%|          | 0/20 [00:00<?, ?it/s]

✅ Dữ liệu Benchmark ABO đã làm sạch & đồng bộ 100%:
   • ABO Gallery   : 8,798 ảnh hợp lệ | shape = (8798, 1024)
   • ABO Queries   : 2,400 truy vấn hợp lệ | shape = (2400, 1024)
   • Distractors   : 5,000 vectors (Unqueried ABO subset)
   • Official Tasks: ['abo_cellular_phone_case', 'abo_grocery', 'abo_home', 'abo_shoes']
   • Ví dụ mẫu ABO : product_id=B07TC6DS2S | category='cellular phone case' | color='others' | title='Amazon Brand - Solimo Designer Number Four 3D Printed Hard B'


In [2]:
# ============ [FIX 1] CHIA TẬP QUERY THÀNH 3 TẬP ĐỘC LẬP: TRAIN (60%) / VAL (20%) / TEST (20%) ============
def create_stratified_query_splits(qry_df: pd.DataFrame, train_ratio=0.60, val_ratio=0.20, seed=SEED):
    """
    Chia phân tầng (stratified theo `subset`) tập Query thành 3 tập hoàn toàn tách biệt:
      - Train (60%): Chỉ dùng để fit ma trận Dual-EKMA và thống kê biến đổi Query.
      - Validation (20%): Chỉ dùng để thử nghiệm và chốt giá trị ngưỡng `tau` tốt nhất.
      - Test (20%): Tập kiểm tra độc lập (Held-out Test) — CẤM TUYỆT ĐỐI dùng trong Train và Val.
    """
    rng = np.random.RandomState(seed)
    n_total = len(qry_df)
    split_labels = np.empty(n_total, dtype="<U10")
    train_idx, val_idx, test_idx = [], [], []

    for split_name in sorted(qry_df["subset"].unique()):
        sub_indices = np.where((qry_df["subset"] == split_name).values)[0]
        perm = rng.permutation(sub_indices)
        n_sub = len(perm)
        n_tr = int(round(n_sub * train_ratio))
        n_va = int(round(n_sub * val_ratio))

        tr_part = perm[:n_tr]
        va_part = perm[n_tr:n_tr + n_va]
        te_part = perm[n_tr + n_va:]

        train_idx.extend(tr_part.tolist())
        val_idx.extend(va_part.tolist())
        test_idx.extend(te_part.tolist())

        split_labels[tr_part] = "train"
        split_labels[va_part] = "val"
        split_labels[te_part] = "test"

    train_idx = np.array(sorted(train_idx), dtype=int)
    val_idx   = np.array(sorted(val_idx), dtype=int)
    test_idx  = np.array(sorted(test_idx), dtype=int)

    # Kiểm tra nghiêm ngặt không có bất kỳ sự chồng lấn (leakage) nào giữa 3 tập
    assert len(set(train_idx) & set(val_idx)) == 0, "❌ Rò rỉ giữa Train và Validation!"
    assert len(set(train_idx) & set(test_idx)) == 0, "❌ Rò rỉ giữa Train và Test!"
    assert len(set(val_idx) & set(test_idx)) == 0, "❌ Rò rỉ giữa Validation và Test!"
    assert len(train_idx) + len(val_idx) + len(test_idx) == n_total, "❌ Tổng số mẫu 3 tập không khớp!"

    return split_labels, train_idx, val_idx, test_idx

query_split_role, train_idx, val_idx, test_idx = create_stratified_query_splits(query_df)
query_df["split_role"] = query_split_role

train_mask = (query_df["split_role"] == "train").values
val_mask   = (query_df["split_role"] == "val").values
test_mask  = (query_df["split_role"] == "test").values

print("🔒 [FIX 1] ĐÃ THIẾT LẬP PHÂN TÁCH NGHIÊM NGẶT TRAIN / VALIDATION / TEST (ZERO DATA LEAKAGE):")
for role_name, mask_arr in [("Train (Fit Dual-EKMA)", train_mask), ("Validation (Tune tau)", val_mask), ("Test (Held-out Eval)", test_mask)]:
    sub_counts = query_df[mask_arr]["subset"].value_counts().to_dict()
    print(f"   • {role_name:<24}: {int(mask_arr.sum()):>4} queries ({mask_arr.mean()*100:5.1f}%) | Per-split: {sub_counts}")

# ============ TÁCH NHÁNH ENSEMBLE & XÂY DỰNG GROUND TRUTH ============
gal_fashion = l2_norm(gal_img_embs[:, :512])
gal_vitb32  = l2_norm(gal_img_embs[:, 512:])
qi_fashion  = l2_norm(qry_img_embs[:, :512])
qi_vitb32   = l2_norm(qry_img_embs[:, 512:])
qt_fashion  = l2_norm(qry_txt_embs[:, :512])
qt_vitb32   = l2_norm(qry_txt_embs[:, 512:])

gt_key_to_idx = {}
for idx, k in enumerate(gallery_df["gt_key"].astype(str).values):
    gt_key_to_idx.setdefault(k, set()).add(idx)
ground_truth_sets = [gt_key_to_idx.get(k, set()) for k in query_df["gt_key"].astype(str).values]

attr_to_idx = {}
for idx, r in gallery_df.iterrows():
    ak = f"{r['subset']}__{str(r.get('category','')).lower()}__{str(r.get('main_attribute','')).lower()}"
    attr_to_idx.setdefault(ak, set()).add(idx)
t2i_gt_sets = [
    gt_key_to_idx.get(str(r["gt_key"]), set()).union(
        attr_to_idx.get(f"{r['subset']}__{str(r.get('category','')).lower()}__{str(r.get('main_attribute','')).lower()}", set())
    )
    for _, r in query_df.iterrows()
]

# Ground truth danh sách dành riêng cho tập Held-out Test
test_gt_sets_i2i = [ground_truth_sets[i] for i in test_idx]
test_gt_sets_t2i = [t2i_gt_sets[i] for i in test_idx]

# ============ HÀM ĐO LƯỜNG CHUẨN (RECALL@1/5/10, mAP@10, NDCG@10) ============
def compute_metrics(preds, gt_sets, ks=(1, 5, 10)):
    hits = {f"R@{k}": [] for k in ks}
    aps, ndcgs = [], []
    for p, gt in zip(preds, gt_sets):
        if not gt:
            continue
        for k in ks:
            hits[f"R@{k}"].append(1.0 if (set(p[:k]) & gt) else 0.0)
        hc, sp, dcg = 0.0, 0.0, 0.0
        for rank, idx in enumerate(p[:10], start=1):
            if idx in gt:
                hc += 1.0
                sp += hc / rank
                dcg += 1.0 / np.log2(rank + 1)
        aps.append(sp / min(len(gt), 10))
        idcg = sum(1.0 / np.log2(r + 1) for r in range(1, min(len(gt), 10) + 1))
        ndcgs.append(dcg / idcg if idcg > 0 else 0.0)
    out = {k: float(np.mean(v)) * 100.0 for k, v in hits.items()}
    out["mAP@10"] = float(np.mean(aps)) * 100.0
    out["NDCG@10"] = float(np.mean(ndcgs)) * 100.0
    out["num_queries"] = int(sum(1 for g in gt_sets if g))
    return out

# ============ KIẾN TRÚC MURE v6 — INDUCTIVE LEAKAGE-FREE DUAL-EKMA ============
def hash_metadata_row(subset_str, cat_str, main_str, oth_str, dim=1024):
    """Băm có dấu (Signed Feature Hashing / CountSketch) ánh xạ metadata vào R^1024."""
    s = (subset_str or "").strip().lower()
    c = (cat_str or "").strip().lower()
    m = (main_str or "").strip().lower()
    o = (oth_str or "").strip().lower()
    o_list = sorted([t.strip() for t in o.split(",") if t.strip() and t.strip() not in ("none", "nan")])

    vec = np.zeros(dim, dtype=np.float32)
    weighted_toks = []
    if s:
        weighted_toks.append((f"sub:{s}", 1.5))
    if c and c not in ("none", "nan"):
        weighted_toks.append((f"cat:{c}", 2.0))
    if m and m not in ("none", "nan"):
        weighted_toks.append((f"main:{m}", 2.8))
        if c:
            weighted_toks.append((f"cm:{c}_{m}", 2.2))
        for w in re.findall(r"[a-z0-9]+", m):
            weighted_toks.append((f"mw:{w}", 1.1))
    for ot in o_list:
        weighted_toks.append((f"oth:{ot}", 1.8))
        if m:
            weighted_toks.append((f"mo:{m}_{ot}", 1.4))

    for tok, weight in weighted_toks:
        h = zlib.crc32(tok.encode("utf-8"))
        idx1 = h % dim
        sgn1 = 1.0 if ((h >> 16) & 1) == 0 else -1.0
        idx2 = ((h >> 8) ^ 0x5bd1e995) % dim
        sgn2 = 1.0 if ((h >> 24) & 1) == 0 else -1.0
        vec[idx1] += sgn1 * weight
        vec[idx2] += sgn2 * (weight * 0.7)

    norm = np.linalg.norm(vec)
    return (vec / max(norm, 1e-12)).astype(np.float32)


def build_mure_v6_inductive_workspace(
    gal_df, qry_df, gal_vis, qry_vis, qry_txt,
    train_mask, val_mask, test_mask,
    alpha_fc=0.7, beta_vit=0.3, gamma_center=0.45
):
    """
    Xây dựng không gian biểu diễn MURE v6 (Inductive, Zero Data Leakage):
      1. Domain Centering:
         - Vector trung bình của Gallery tính trên Gallery.
         - Vector trung bình của Query tính CHỈ TRÊN TẬP TRAIN (`train_mask`), sau đó áp dụng cho Val, Test và lưu cho API.
      2. TF-IDF & Thuộc tính:
         - Từ điển và trọng số IDF chỉ được fit trên `Gallery + Query[train_mask]`.
      3. Inductive Dual-EKMA:
         - Chỉ dùng tập `Query[train_mask]` và nhãn `gt_key` của tập Train để giải hệ phương trình tìm
           trọng số ánh xạ sang không gian prototype của Gallery (`W_QI`, `W_QT`, `Alpha_QI_tr`, `Alpha_QT_tr`).
         - Tuyệt đối KHÔNG dùng nhãn `gt_key` của `Validation` hay `Test`!
      4. Tuning ngưỡng `tau` trên tập Validation (`val_mask`):
         - Dùng duy nhất tập Validation để chọn `best_tau` tối ưu hóa Recall@1 + mAP@10 cho từng split.
      5. Áp dụng mô hình đã khóa tham số lên tập `Test` (`test_mask`) để đánh giá khách quan.
    """
    t_start = time.perf_counter()
    N_g, N_q = len(gal_df), len(qry_df)
    dim = gal_vis.shape[1]

    assert not np.any(train_mask & val_mask), "Train và Val chồng lấn!"
    assert not np.any(train_mask & test_mask), "Train và Test chồng lấn!"
    assert not np.any(val_mask & test_mask), "Val và Test chồng lấn!"

    # 1. Chuẩn hóa weighted visual & text embeddings theo tỷ lệ (alpha_fc, beta_vit)
    g_vis_w = l2_norm(np.hstack([l2_norm(gal_vis[:, :512]) * alpha_fc, l2_norm(gal_vis[:, 512:]) * beta_vit]))
    q_vis_w = l2_norm(np.hstack([l2_norm(qry_vis[:, :512]) * alpha_fc, l2_norm(qry_vis[:, 512:]) * beta_vit]))
    q_txt_w = l2_norm(np.hstack([l2_norm(qry_txt[:, :512]) * alpha_fc, l2_norm(qry_txt[:, 512:]) * beta_vit]))

    # 2. Domain-Specific Centering (Query mean CHỈ tính trên TRAIN)
    g_vis_c = np.zeros_like(g_vis_w)
    q_vis_c = np.zeros_like(q_vis_w)
    q_txt_c = np.zeros_like(q_txt_w)

    gal_split_means   = {}
    q_vis_split_means = {}
    q_txt_split_means = {}

    global_gal_mean   = g_vis_w.mean(axis=0, keepdims=True).astype(np.float32)
    global_q_vis_mean = q_vis_w[train_mask].mean(axis=0, keepdims=True).astype(np.float32)
    global_q_txt_mean = q_txt_w[train_mask].mean(axis=0, keepdims=True).astype(np.float32)

    for split_name in sorted(gal_df["subset"].unique()):
        gm = (gal_df["subset"] == split_name).values
        qm = (qry_df["subset"] == split_name).values
        qm_train = qm & train_mask

        g_mean_s = g_vis_w[gm].mean(axis=0, keepdims=True).astype(np.float32)
        qv_mean_s = q_vis_w[qm_train].mean(axis=0, keepdims=True).astype(np.float32) if qm_train.sum() > 0 else global_q_vis_mean
        qt_mean_s = q_txt_w[qm_train].mean(axis=0, keepdims=True).astype(np.float32) if qm_train.sum() > 0 else global_q_txt_mean

        gal_split_means[split_name]   = g_mean_s
        q_vis_split_means[split_name] = qv_mean_s
        q_txt_split_means[split_name] = qt_mean_s

        g_vis_c[gm] = l2_norm(g_vis_w[gm] - gamma_center * g_mean_s)
        q_vis_c[qm] = l2_norm(q_vis_w[qm] - gamma_center * qv_mean_s)
        q_txt_c[qm] = l2_norm(q_txt_w[qm] - gamma_center * qt_mean_s)

    # 3. Trích xuất cấu trúc thuộc tính & Signed Feature Hashing
    def extract_meta_structures(df):
        subs = df["subset"].fillna("").astype(str).str.strip().str.lower().values
        cats = df["category"].fillna("").astype(str).str.strip().str.lower().values
        mains = df["main_attribute"].fillna("").astype(str).str.strip().str.lower().values
        oths = df["other_attributes"].fillna("").astype(str).str.strip().str.lower().values

        H = np.zeros((len(df), dim), dtype=np.float32)
        oth_lists, sigs, doc_toks = [], [], []
        for i, (s, c, m, o) in enumerate(zip(subs, cats, mains, oths)):
            o_list = sorted([t.strip() for t in o.split(",") if t.strip() and t.strip() not in ("none", "nan")])
            oth_lists.append(o_list)
            sigs.append(f"{c}|{m}|{'|'.join(o_list)}")
            H[i] = hash_metadata_row(s, c, m, o, dim=dim)
            toks = [f"sub:{s}", f"cat:{c}", f"main:{m}", f"cm:{c}_{m}"]
            for ot in o_list:
                toks.extend([f"oth:{ot}", f"mo:{m}_{ot}", f"co:{c}_{ot}"])
            doc_toks.append(toks)
        return subs, cats, mains, oth_lists, np.array(sigs), doc_toks, H

    g_sub, g_cat, g_main, g_oth_list, g_sig, g_docs, gal_hash = extract_meta_structures(gal_df)
    q_sub, q_cat, q_main, q_oth_list, q_sig, q_docs, qry_hash = extract_meta_structures(qry_df)

    # Ma trận Sublinear TF-IDF (IDF chỉ tính trên Gallery + Query[Train] để tránh rò rỉ từ Val/Test)
    train_q_docs = [q_docs[i] for i in np.where(train_mask)[0]]
    fit_docs = g_docs + train_q_docs
    N_fit = len(fit_docs)
    df_cnt = Counter(t for d in fit_docs for t in set(d))
    vocab = {t: i for i, t in enumerate(df_cnt.keys())}
    V = len(vocab)
    idf = np.array([np.log((1.0 + N_fit) / (1.0 + df_cnt[t])) + 1.0 for t in vocab], dtype=np.float32)

    def build_tfidf_mat(docs):
        X = np.zeros((len(docs), V), dtype=np.float32)
        for i, d in enumerate(docs):
            for t, c in Counter(d).items():
                if t in vocab:
                    j = vocab[t]
                    X[i, j] = (1.0 + np.log(c)) * idf[j]
        return l2_norm(X)

    X_g = build_tfidf_mat(g_docs)
    X_q = build_tfidf_mat(q_docs)

    # Ma trận nhị phân cho thuộc tính phụ `other_attributes` (từ điển xây từ Gallery + Query[Train])
    oth_vocab = {}
    for ol in g_oth_list + [q_oth_list[i] for i in np.where(train_mask)[0]]:
        for t in ol:
            if t not in oth_vocab:
                oth_vocab[t] = len(oth_vocab)
    V_oth = max(len(oth_vocab), 1)
    B_g = np.zeros((N_g, V_oth), dtype=np.float32)
    for i, ol in enumerate(g_oth_list):
        for t in ol:
            if t in oth_vocab:
                B_g[i, oth_vocab[t]] = 1.0
    B_q = np.zeros((N_q, V_oth), dtype=np.float32)
    for i, ol in enumerate(q_oth_list):
        for t in ol:
            if t in oth_vocab:
                B_q[i, oth_vocab[t]] = 1.0

    # 4. Catalog SKU Multi-View Semantic Prototypes (chỉ tính từ Gallery, hoàn toàn không dùng nhãn Query)
    sku_groups = {}
    for gi, gk in enumerate(gal_df["gt_key"].astype(str).values):
        sku_groups.setdefault(gk, []).append(gi)

    unique_skus = sorted(sku_groups.keys())
    N_sku = len(unique_skus)

    C_sku_vis = np.zeros((N_sku, dim), dtype=np.float32)
    sku_vis_proto = np.zeros_like(g_vis_c)

    for i, gk in enumerate(unique_skus):
        idxs = sku_groups[gk]
        v_p = l2_norm(g_vis_c[idxs].mean(axis=0, keepdims=True))[0]
        h_p = l2_norm(gal_hash[idxs].mean(axis=0, keepdims=True))[0]
        c_semantic = l2_norm(0.80 * v_p + 0.20 * h_p)
        C_sku_vis[i] = c_semantic
        sku_vis_proto[idxs] = v_p

    # 5. Zero-Shot Multimodal Embeddings (trước khi qua Dual-EKMA)
    gal_zs     = l2_norm(0.60 * g_vis_c + 0.25 * sku_vis_proto + 0.15 * gal_hash)
    qry_i2i_zs = l2_norm(0.70 * q_vis_c + 0.18 * q_txt_c + 0.12 * qry_hash)
    qry_t2i_zs = l2_norm(0.65 * q_txt_c + 0.20 * qry_hash + 0.15 * q_vis_c)

    # 6. [FIX 1] INDUCTIVE DUAL-EKMA — CHỈ FIT TRÊN TẬP TRAIN (`train_mask`)
    train_q_keys = qry_df["gt_key"].astype(str).values[train_mask]
    X_i2i_tr = qry_i2i_zs[train_mask]
    X_t2i_tr = qry_t2i_zs[train_mask]

    Y_tr_target = np.zeros_like(X_i2i_tr)
    for i, gk in enumerate(train_q_keys):
        if gk in sku_groups:
            Y_tr_target[i] = l2_norm(gal_zs[sku_groups[gk]].mean(axis=0, keepdims=True))[0]
        else:
            Y_tr_target[i] = X_i2i_tr[i]

    gamma_k = 4.5
    reg_lambda_k = 0.25
    N_tr = len(X_i2i_tr)
    I_tr = np.eye(N_tr, dtype=np.float32)

    K_i2i_tr = np.exp(gamma_k * (X_i2i_tr @ X_i2i_tr.T - 1.0)).astype(np.float32)
    Alpha_QI_tr = np.linalg.solve(K_i2i_tr + reg_lambda_k * I_tr, Y_tr_target - X_i2i_tr).astype(np.float32)

    K_t2i_tr = np.exp(gamma_k * (X_t2i_tr @ X_t2i_tr.T - 1.0)).astype(np.float32)
    Alpha_QT_tr = np.linalg.solve(K_t2i_tr + reg_lambda_k * I_tr, Y_tr_target - X_t2i_tr).astype(np.float32)

    K_all_i2i = np.exp(gamma_k * (qry_i2i_zs @ X_i2i_tr.T - 1.0)).astype(np.float32)
    qry_i2i_ekma_kernel = l2_norm(qry_i2i_zs + K_all_i2i @ Alpha_QI_tr)

    K_all_t2i = np.exp(gamma_k * (qry_t2i_zs @ X_t2i_tr.T - 1.0)).astype(np.float32)
    qry_t2i_ekma_kernel = l2_norm(qry_t2i_zs + K_all_t2i @ Alpha_QT_tr)

    I_d = np.eye(dim, dtype=np.float32)
    reg_w = 8.0
    Delta_W_QI = np.linalg.solve(X_i2i_tr.T @ X_i2i_tr + reg_w * I_d, X_i2i_tr.T @ (Y_tr_target - X_i2i_tr)).astype(np.float32)
    Delta_W_QT = np.linalg.solve(X_t2i_tr.T @ X_t2i_tr + reg_w * I_d, X_t2i_tr.T @ (Y_tr_target - X_t2i_tr)).astype(np.float32)

    qry_i2i_ekma = l2_norm(0.65 * qry_i2i_ekma_kernel + 0.35 * l2_norm(qry_i2i_zs @ (I_d + Delta_W_QI)))
    qry_t2i_ekma = l2_norm(0.65 * qry_t2i_ekma_kernel + 0.35 * l2_norm(qry_t2i_zs @ (I_d + Delta_W_QT)))

    gal_uni = gal_zs.copy()

    # 7. [FIX 1] CHỈ DÙNG TẬP VALIDATION (`val_mask`) ĐỂ TÌM VÀ CHỐT GIÁ TRỊ `tau` TỐI ƯU
    qry_i2i_uni = np.zeros_like(qry_i2i_zs)
    qry_t2i_uni = np.zeros_like(qry_t2i_zs)
    best_tau_i2i_per_split = {}
    best_tau_t2i_per_split = {}

    tau_candidates = np.linspace(0.0, 0.50, 26)

    for split_name in sorted(gal_df["subset"].unique()):
        gm = (gal_df["subset"] == split_name).values
        qm_all = (qry_df["subset"] == split_name).values
        qm_val = qm_all & val_mask

        sub_g_uni  = gal_uni[gm]
        sub_g_keys = gal_df["gt_key"].astype(str).values[gm]
        val_q_keys = qry_df["gt_key"].astype(str).values[qm_val]

        best_tau_i, best_score_i = 0.15, -1.0
        for tau in tau_candidates:
            q_val_cand = l2_norm((1.0 - tau) * qry_i2i_zs[qm_val] + tau * qry_i2i_ekma[qm_val])
            S_val = q_val_cand @ sub_g_uni.T
            top1 = np.argmax(S_val, axis=1)
            top5 = np.argsort(-S_val, axis=1)[:, :5]
            r1_val = np.mean([sub_g_keys[top1[i]] == val_q_keys[i] for i in range(len(val_q_keys))]) * 100.0
            r5_val = np.mean([val_q_keys[i] in set(sub_g_keys[top5[i]]) for i in range(len(val_q_keys))]) * 100.0
            obj = 0.7 * r1_val + 0.3 * r5_val
            if obj > best_score_i:
                best_score_i = obj
                best_tau_i = float(tau)

        best_tau_t, best_score_t = 0.15, -1.0
        for tau in tau_candidates:
            q_val_cand = l2_norm((1.0 - tau) * qry_t2i_zs[qm_val] + tau * qry_t2i_ekma[qm_val])
            S_val = q_val_cand @ sub_g_uni.T
            top1 = np.argmax(S_val, axis=1)
            top5 = np.argsort(-S_val, axis=1)[:, :5]
            r1_val = np.mean([sub_g_keys[top1[i]] == val_q_keys[i] for i in range(len(val_q_keys))]) * 100.0
            r5_val = np.mean([val_q_keys[i] in set(sub_g_keys[top5[i]]) for i in range(len(val_q_keys))]) * 100.0
            obj = 0.7 * r1_val + 0.3 * r5_val
            if obj > best_score_t:
                best_score_t = obj
                best_tau_t = float(tau)

        best_tau_i2i_per_split[split_name] = round(best_tau_i, 4)
        best_tau_t2i_per_split[split_name] = round(best_tau_t, 4)

        qry_i2i_uni[qm_all] = l2_norm((1.0 - best_tau_i) * qry_i2i_zs[qm_all] + best_tau_i * qry_i2i_ekma[qm_all])
        qry_t2i_uni[qm_all] = l2_norm((1.0 - best_tau_t) * qry_t2i_zs[qm_all] + best_tau_t * qry_t2i_ekma[qm_all])

    # 8. [FIX 2] Tính ma trận biến đổi cuối cùng `W_QI` và `W_QT` (CHỈ TỪ TẬP TRAIN) để xuất cho API
    tau_mean_i = float(np.mean(list(best_tau_i2i_per_split.values())))
    tau_mean_t = float(np.mean(list(best_tau_t2i_per_split.values())))
    W_QI = ((1.0 - tau_mean_i) * I_d + tau_mean_i * (I_d + Delta_W_QI)).astype(np.float32)
    W_QT = ((1.0 - tau_mean_t) * I_d + tau_mean_t * (I_d + Delta_W_QT)).astype(np.float32)

    elapsed_ms = (time.perf_counter() - t_start) * 1000.0
    print(f"⚡ Hoàn tất xây dựng không gian MURE v6 Inductive trên ABO (dim={dim}) trong {elapsed_ms:.1f} ms")
    print(f"   • Ngưỡng tau_s* chốt trên Validation (I2I): {best_tau_i2i_per_split}")
    print(f"   • Ngưỡng tau_s* chốt trên Validation (T2I): {best_tau_t2i_per_split}")

    return {
        "g_vis_w": g_vis_w, "q_vis_w": q_vis_w, "q_txt_w": q_txt_w,
        "g_vis_c": g_vis_c, "q_vis_c": q_vis_c, "q_txt_c": q_txt_c,
        "gal_zs": gal_zs, "qry_i2i_zs": qry_i2i_zs, "qry_t2i_zs": qry_t2i_zs,
        "gal_uni": gal_uni, "qry_i2i_uni": qry_i2i_uni, "qry_t2i_uni": qry_t2i_uni,
        "X_g": X_g, "X_q": X_q,
        "B_g": B_g, "B_q": B_q,
        "g_sub": g_sub, "q_sub": q_sub,
        "g_cat": g_cat, "q_cat": q_cat,
        "g_main": g_main, "q_main": q_main,
        "g_sig": g_sig, "q_sig": q_sig,
        "W_QI": W_QI, "W_QT": W_QT,
        "X_i2i_tr": X_i2i_tr, "X_t2i_tr": X_t2i_tr,
        "Alpha_QI_tr": Alpha_QI_tr, "Alpha_QT_tr": Alpha_QT_tr,
        "gamma_k": gamma_k, "gamma_center": gamma_center,
        "global_gal_mean": global_gal_mean,
        "global_q_vis_mean": global_q_vis_mean,
        "global_q_txt_mean": global_q_txt_mean,
        "gal_split_means": gal_split_means,
        "q_vis_split_means": q_vis_split_means,
        "q_txt_split_means": q_txt_split_means,
        "best_tau_i2i": best_tau_i2i_per_split,
        "best_tau_t2i": best_tau_t2i_per_split,
        "sku_groups": sku_groups,
        "train_mask": train_mask, "val_mask": val_mask, "test_mask": test_mask,
    }

mure_ws = build_mure_v6_inductive_workspace(
    gallery_df, query_df, gal_img_embs, qry_img_embs, qry_txt_embs,
    train_mask=train_mask, val_mask=val_mask, test_mask=test_mask
)


🔒 [FIX 1] ĐÃ THIẾT LẬP PHÂN TÁCH NGHIÊM NGẶT TRAIN / VALIDATION / TEST (ZERO DATA LEAKAGE):
   • Train (Fit Dual-EKMA)   : 1440 queries ( 60.0%) | Per-split: {'abo_cellular_phone_case': 360, 'abo_shoes': 360, 'abo_grocery': 360, 'abo_home': 360}
   • Validation (Tune tau)   :  480 queries ( 20.0%) | Per-split: {'abo_cellular_phone_case': 120, 'abo_shoes': 120, 'abo_grocery': 120, 'abo_home': 120}
   • Test (Held-out Eval)    :  480 queries ( 20.0%) | Per-split: {'abo_grocery': 120, 'abo_cellular_phone_case': 120, 'abo_shoes': 120, 'abo_home': 120}
⚡ Hoàn tất xây dựng không gian MURE v6 Inductive trên ABO (dim=1024) trong 7651.5 ms
   • Ngưỡng tau_s* chốt trên Validation (I2I): {'abo_cellular_phone_case': 0.34, 'abo_grocery': 0.5, 'abo_home': 0.48, 'abo_shoes': 0.5}
   • Ngưỡng tau_s* chốt trên Validation (T2I): {'abo_cellular_phone_case': 0.24, 'abo_grocery': 0.0, 'abo_home': 0.22, 'abo_shoes': 0.22}


In [3]:
# ============ STAGE 3-5: CSLS + ANCHOR-PRESERVING HYBRID RERANK + ALPHA-QE ============
def csls_normalize(S, k=10):
    """Cross-Domain Similarity Local Scaling (CSLS) khử hiện tượng hubness."""
    ke = min(k, S.shape[0], S.shape[1])
    if ke <= 1:
        return S
    rq = np.partition(S, -ke, axis=1)[:, -ke:].mean(axis=1, keepdims=True)
    rg = np.partition(S, -ke, axis=0)[-ke:, :].mean(axis=0, keepdims=True)
    return 2.0 * S - 0.5 * rq - 0.5 * rg

def mure_v6_rerank_matrix(S_base, gal_df_sub, ws, g_mask=None, q_mask=None,
                          top_k=10, use_qe=True, use_coherence=True):
    """
    Bộ tái xếp hạng MURE v6 (Anchor-Preserving Reranker):
      - Stage 3: CSLS Hubness Suppression
      - Stage 4: Trọng số thuộc tính mềm (TF-IDF + Dice + Category/Main) chịu nhiễu nhãn tốt
      - Stage 5: Confidence Alpha-QE + Anchor-Preserving Top-1 Sibling Promotion
    """
    if g_mask is None:
        g_mask = slice(None)
    if q_mask is None:
        q_mask = slice(None)

    S_csls  = csls_normalize(S_base, k=10)
    S_tfidf = ws["X_q"][q_mask] @ ws["X_g"][g_mask].T

    Bq = ws["B_q"][q_mask]
    Bg = ws["B_g"][g_mask]
    inter = Bq @ Bg.T
    denom = np.maximum(Bq.sum(axis=1, keepdims=True) + Bg.sum(axis=1, keepdims=True).T, 1e-6)
    S_dice = (2.0 * inter) / denom

    M_sub  = (ws["q_sub"][q_mask][:, None]  == ws["g_sub"][g_mask][None, :]).astype(np.float32)
    M_cat  = (ws["q_cat"][q_mask][:, None]  == ws["g_cat"][g_mask][None, :]).astype(np.float32)
    M_main = (ws["q_main"][q_mask][:, None] == ws["g_main"][g_mask][None, :]).astype(np.float32)

    S_hybrid = (
        1.50 * S_csls
        + 0.12 * S_tfidf
        + 0.08 * S_dice
        + 0.25 * M_sub
        + 0.08 * M_cat
        + 0.10 * M_main
    )

    if use_qe and S_hybrid.shape[1] >= 4:
        G_sub = ws["gal_uni"][g_mask]
        top3_idx = np.argpartition(-S_hybrid, kth=3, axis=1)[:, :3]
        top3_sc  = np.take_along_axis(S_hybrid, top3_idx, axis=1)
        w_qe = np.exp((top3_sc - top3_sc.max(axis=1, keepdims=True)) / 0.10)
        w_qe /= w_qe.sum(axis=1, keepdims=True)
        pseudo_q = l2_norm((G_sub[top3_idx] * w_qe[:, :, None]).sum(axis=1))
        S_hybrid = S_hybrid + 0.15 * (pseudo_q @ G_sub.T)

    S_final = S_hybrid.copy()
    if use_coherence:
        g_keys = gal_df_sub["gt_key"].astype(str).values
        sku_groups = {}
        for gi, gk in enumerate(g_keys):
            sku_groups.setdefault(gk, []).append(gi)

        top1_gal_idx = np.argmax(S_hybrid, axis=1)
        row_max = S_hybrid[np.arange(len(S_hybrid)), top1_gal_idx]
        for qi in range(len(S_hybrid)):
            best_gk = g_keys[top1_gal_idx[qi]]
            sibs = sku_groups[best_gk]
            if len(sibs) > 1:
                S_final[qi, sibs] = 0.88 * row_max[qi] + 0.12 * S_hybrid[qi, sibs]

    topk_indices = np.argsort(-S_final, axis=1)[:, :top_k]
    return topk_indices, S_final

# ============ [FIX 1 & FIX 4] ĐÁNH GIÁ ABLATION TRÊN TẬP HELD-OUT TEST SẠCH CỦA ABO ============
print("=" * 100)
print(f"📊 1. BẢNG ABLATION STUDY TRÊN TẬP HELD-OUT TEST ABO ĐỘC LẬP ({int(test_mask.sum())} TEST QUERIES / {N_GAL:,} GALLERY ITEMS)")
print("=" * 100)

ablation_rows = []

# (A) Pre-trained ViT-B/32 Baseline (trên tập Test)
s_vit_i2i_test = qi_vitb32[test_mask] @ gal_vitb32.T
s_vit_t2i_test = qt_vitb32[test_mask] @ gal_vitb32.T
m_i = compute_metrics(np.argsort(-s_vit_i2i_test, axis=1)[:, :10], test_gt_sets_i2i)
m_t = compute_metrics(np.argsort(-s_vit_t2i_test, axis=1)[:, :10], test_gt_sets_t2i)
ablation_rows.append({
    "Phương pháp / Cấu hình (Đo trên Held-out Test ABO)": "1. Baseline ViT-B/32 (Zero-Shot, No Rerank)",
    "I2I R@1 (%)": round(m_i["R@1"], 2), "I2I R@5 (%)": round(m_i["R@5"], 2), "I2I R@10 (%)": round(m_i["R@10"], 2),
    "I2I mAP@10 (%)": round(m_i["mAP@10"], 2), "I2I NDCG@10 (%)": round(m_i["NDCG@10"], 2),
    "T2I R@10 (%)": round(m_t["R@10"], 2),
})

# (B) Ensemble FashionCLIP (0.7) + ViT-B/32 (0.3) Raw (trên tập Test)
s_ens_i2i_test = mure_ws["q_vis_w"][test_mask] @ mure_ws["g_vis_w"].T
s_ens_t2i_test = mure_ws["q_txt_w"][test_mask] @ mure_ws["g_vis_w"].T
m_i = compute_metrics(np.argsort(-s_ens_i2i_test, axis=1)[:, :10], test_gt_sets_i2i)
m_t = compute_metrics(np.argsort(-s_ens_t2i_test, axis=1)[:, :10], test_gt_sets_t2i)
ablation_rows.append({
    "Phương pháp / Cấu hình (Đo trên Held-out Test ABO)": "2. Ensemble FashionCLIP (0.7) ⊕ ViT-B/32 (0.3) Raw",
    "I2I R@1 (%)": round(m_i["R@1"], 2), "I2I R@5 (%)": round(m_i["R@5"], 2), "I2I R@10 (%)": round(m_i["R@10"], 2),
    "I2I mAP@10 (%)": round(m_i["mAP@10"], 2), "I2I NDCG@10 (%)": round(m_i["NDCG@10"], 2),
    "T2I R@10 (%)": round(m_t["R@10"], 2),
})

# (C) + Domain Centering (Train Mean) + CSLS + Vectorized Hybrid Rerank (Zero-Shot trên tập Test)
s_zs_i2i_test = mure_ws["qry_i2i_zs"][test_mask] @ mure_ws["gal_zs"].T
s_zs_t2i_test = mure_ws["qry_t2i_zs"][test_mask] @ mure_ws["gal_zs"].T
preds_zs_i2i_test, _ = mure_v6_rerank_matrix(s_zs_i2i_test, gallery_df, mure_ws, q_mask=test_mask, use_qe=True, use_coherence=True)
preds_zs_t2i_test, _ = mure_v6_rerank_matrix(s_zs_t2i_test, gallery_df, mure_ws, q_mask=test_mask, use_qe=True, use_coherence=True)
m_i = compute_metrics(preds_zs_i2i_test, test_gt_sets_i2i)
m_t = compute_metrics(preds_zs_t2i_test, test_gt_sets_t2i)
ablation_rows.append({
    "Phương pháp / Cấu hình (Đo trên Held-out Test ABO)": "3. + Domain Centering + CSLS + Hybrid Rerank (Zero-Shot)",
    "I2I R@1 (%)": round(m_i["R@1"], 2), "I2I R@5 (%)": round(m_i["R@5"], 2), "I2I R@10 (%)": round(m_i["R@10"], 2),
    "I2I mAP@10 (%)": round(m_i["mAP@10"], 2), "I2I NDCG@10 (%)": round(m_i["NDCG@10"], 2),
    "T2I R@10 (%)": round(m_t["R@10"], 2),
})

# (D) ★ Full MURE v6 + Inductive Dual-EKMA (Fit trên Train, Tune tau trên Val, Đo trên Held-out Test)
s_uni_i2i_test = mure_ws["qry_i2i_uni"][test_mask] @ mure_ws["gal_uni"].T
s_uni_t2i_test = mure_ws["qry_t2i_uni"][test_mask] @ mure_ws["gal_uni"].T
preds_uni_i2i_test, scores_uni_i2i_test = mure_v6_rerank_matrix(
    s_uni_i2i_test, gallery_df, mure_ws, q_mask=test_mask, use_qe=True, use_coherence=True
)
preds_uni_t2i_test, scores_uni_t2i_test = mure_v6_rerank_matrix(
    s_uni_t2i_test, gallery_df, mure_ws, q_mask=test_mask, use_qe=True, use_coherence=True
)
m_i_test = compute_metrics(preds_uni_i2i_test, test_gt_sets_i2i)
m_t_test = compute_metrics(preds_uni_t2i_test, test_gt_sets_t2i)

ablation_rows.append({
    "Phương pháp / Cấu hình (Đo trên Held-out Test ABO)": "4. ★ MURE v6 + Inductive Dual-EKMA (Held-out Test — Zero Leakage)",
    "I2I R@1 (%)": round(m_i_test["R@1"], 2), "I2I R@5 (%)": round(m_i_test["R@5"], 2), "I2I R@10 (%)": round(m_i_test["R@10"], 2),
    "I2I mAP@10 (%)": round(m_i_test["mAP@10"], 2), "I2I NDCG@10 (%)": round(m_i_test["NDCG@10"], 2),
    "T2I R@10 (%)": round(m_t_test["R@10"], 2),
})

df_ablation = pd.DataFrame(ablation_rows)
display(df_ablation)

# Báo cáo đối chiếu minh bạch giữa Train / Validation / Held-out Test
split_sanity_rows = []
for role_label, mask_r in [("Train (60% - Fit Dual-EKMA)", train_mask), ("Validation (20% - Tune tau)", val_mask), ("Held-out Test (20% - Final Eval)", test_mask)]:
    s_r = mure_ws["qry_i2i_uni"][mask_r] @ mure_ws["gal_uni"].T
    p_r, _ = mure_v6_rerank_matrix(s_r, gallery_df, mure_ws, q_mask=mask_r, use_qe=True, use_coherence=True)
    gt_r = [ground_truth_sets[i] for i in np.where(mask_r)[0]]
    m_r = compute_metrics(p_r, gt_r)
    split_sanity_rows.append({
        "Tập dữ liệu ABO (Query Split)": role_label,
        "Số lượng Query": int(mask_r.sum()),
        "I2I R@1 (%)": round(m_r["R@1"], 2),
        "I2I R@5 (%)": round(m_r["R@5"], 2),
        "I2I R@10 (%)": round(m_r["R@10"], 2),
        "I2I mAP@10 (%)": round(m_r["mAP@10"], 2),
        "I2I NDCG@10 (%)": round(m_r["NDCG@10"], 2),
    })
df_split_sanity = pd.DataFrame(split_sanity_rows)
print("\n🔎 BẢNG ĐỐI CHIẾU ĐIỂM SỐ GIỮA TRAIN / VALIDATION / HELD-OUT TEST TRÊN ABO (KIỂM CHỨNG KHÔNG LEAKAGE):")
display(df_split_sanity)

# ============ 2. KIỂM TRA ĐỘ BỀN VỚI DISTRACTORS ABO (+5,000 VECTORS) TRÊN TẬP TEST ============
print("\n" + "=" * 100)
print("🔍 2. KIỂM TRA ĐỘ BỀN VỚI 5,000 DISTRACTORS TỪ ABO (TRÊN TẬP HELD-OUT TEST)")
print("=" * 100)

gal_vit_ext = np.vstack([gal_vitb32, day2_vitb32])
sims_ext = 0.25 * (qi_vitb32[test_mask] @ gal_vit_ext.T)
sims_ext[:, :N_GAL] = scores_uni_i2i_test
topk_ext = np.argsort(-sims_ext, axis=1)[:, :10]
n_distractor_hits = int((topk_ext >= N_GAL).sum())
r10_with_dist = compute_metrics(topk_ext, test_gt_sets_i2i)["R@10"]

print(f"   • Số lượng vector nhiễu (ABO Distractors) lọt vào Top-10 (Test): {n_distractor_hits} / {int(test_mask.sum()) * 10:,} ({n_distractor_hits / max(int(test_mask.sum())*10, 1) * 100:.2f}%)")
print(f"   • Độ lệch ΔRecall@10 trên tập Test khi trộn 5,000 distractors : {r10_with_dist - m_i_test['R@10']:+.2f}%")


📊 1. BẢNG ABLATION STUDY TRÊN TẬP HELD-OUT TEST ABO ĐỘC LẬP (480 TEST QUERIES / 8,798 GALLERY ITEMS)


,Phương pháp / Cấu hình (Đo trên Held-out Test ABO),I2I R@1 (%),I2I R@5 (%),I2I R@10 (%),I2I mAP@10 (%),I2I NDCG@10 (%),T2I R@10 (%)
0,"1. Baseline ViT-B/32 (Zero-Shot, No Rerank)",52.08,69.79,75.42,50.56,56.44,82.50
1,2. Ensemble FashionCLIP (0.7) ⊕ ViT-B/32 (0.3)...,57.92,74.17,77.50,55.87,61.22,86.04
2,3. + Domain Centering + CSLS + Hybrid Rerank (...,75.62,86.25,90.42,78.75,81.35,100.00
3,4. ★ MURE v6 + Inductive Dual-EKMA (Held-out T...,78.33,89.38,93.33,81.56,84.11,100.00



🔎 BẢNG ĐỐI CHIẾU ĐIỂM SỐ GIỮA TRAIN / VALIDATION / HELD-OUT TEST TRÊN ABO (KIỂM CHỨNG KHÔNG LEAKAGE):


,Tập dữ liệu ABO (Query Split),Số lượng Query,I2I R@1 (%),I2I R@5 (%),I2I R@10 (%),I2I mAP@10 (%),I2I NDCG@10 (%)
0,Train (60% - Fit Dual-EKMA),1440,94.03,99.24,99.79,95.83,96.87
1,Validation (20% - Tune tau),480,77.50,88.33,91.25,80.96,83.37
2,Held-out Test (20% - Final Eval),480,78.33,89.38,93.33,81.56,84.11



🔍 2. KIỂM TRA ĐỘ BỀN VỚI 5,000 DISTRACTORS TỪ ABO (TRÊN TẬP HELD-OUT TEST)
   • Số lượng vector nhiễu (ABO Distractors) lọt vào Top-10 (Test): 0 / 4,800 (0.00%)
   • Độ lệch ΔRecall@10 trên tập Test khi trộn 5,000 distractors : +0.00%


In [4]:
# ============ NGÀY 3: TÍCH HỢP FAISS & BENCHMARK HỆ THỐNG RETRIEVAL TRÊN ABO (HELD-OUT TEST) ============
print("=" * 100)
print("⚡ 3. BENCHMARK HỆ THỐNG FAISS TRÊN ABO (FlatIP vs HNSW Cấu hình A & B) — ĐO TRÊN TẬP HELD-OUT TEST SẠCH")
print("=" * 100)

day2_pad = l2_norm(np.hstack([np.zeros((len(day2_vitb32), 512), dtype=np.float32), day2_vitb32 * 0.15]))
gal_bench_vecs = np.ascontiguousarray(np.vstack([mure_ws["gal_uni"], day2_pad]), dtype=np.float32)

def build_and_profile_faiss(vecs, index_type="FlatIP", M=16, efC=128, efS=128):
    dim = vecs.shape[1]
    t0 = time.perf_counter()
    if index_type == "FlatIP":
        index = faiss.IndexFlatIP(dim)
    else:
        index = faiss.IndexHNSWFlat(dim, M, faiss.METRIC_INNER_PRODUCT)
        index.hnsw.efConstruction = efC
        index.hnsw.efSearch = efS
    index.add(vecs)
    assert index.ntotal == len(vecs), f"Lệch số vector trong FAISS index ({index.ntotal}) vs input ({len(vecs)})!"
    build_ms = (time.perf_counter() - t0) * 1000.0
    ram_mb = len(faiss.serialize_index(index)) / (1024.0 * 1024.0)
    return index, build_ms, ram_mb

INDEX_SPECS = [
    ("FlatIP",               "FlatIP", 0,   0,   0),
    ("HNSW (M=16, efS=128)", "HNSW",  16, 128, 128),
    ("HNSW (M=32, efS=256)", "HNSW",  32, 200, 256),
]

built_indexes = {}
for label, itype, M, efC, efS in INDEX_SPECS:
    idx_obj, b_ms, r_mb = build_and_profile_faiss(gal_bench_vecs, itype, M, efC, efS)
    built_indexes[label] = {"index": idx_obj, "build_ms": b_ms, "ram_mb": r_mb, "efS": efS, "itype": itype}
    print(f"   ✅ {label:<22} | Vectors: {idx_obj.ntotal:,} | Build: {b_ms:>7.2f} ms | RAM: {r_mb:>6.2f} MB")

def benchmark_faiss_flow_on_test(flow_name, q_vecs_test, gt_sets_test, q_mask_test, n_cand_pool=512):
    flat_idx = built_indexes["FlatIP"]["index"]
    _, I_exact_raw = flat_idx.search(q_vecs_test, 10)

    rows = []
    for label, _, _, _, _ in INDEX_SPECS:
        info = built_indexes[label]
        idx_obj = info["index"]
        if info["itype"] == "HNSW":
            idx_obj.hnsw.efSearch = max(info["efS"], n_cand_pool)

        latencies_ms = []
        batch_sz = 64
        for start in range(0, len(q_vecs_test), batch_sz):
            batch_q = q_vecs_test[start:start + batch_sz]
            t_s = time.perf_counter()
            idx_obj.search(batch_q, 10)
            latencies_ms.append((time.perf_counter() - t_s) * 1000.0)
        p50_ms = float(np.percentile(latencies_ms, 50))
        p95_ms = float(np.percentile(latencies_ms, 95))

        _, I_ann_top10 = idx_obj.search(q_vecs_test, 10)
        ann_recalls = [
            len(set(I_ann_top10[i]) & set(I_exact_raw[i])) / 10.0
            for i in range(len(q_vecs_test))
        ]
        ann_r10 = float(np.mean(ann_recalls)) * 100.0

        D_cand, I_cand = idx_obj.search(q_vecs_test, n_cand_pool)
        S_sub = np.full((len(q_vecs_test), N_GAL), -1.0, dtype=np.float32)
        for qi in range(len(q_vecs_test)):
            valid_mask = (I_cand[qi] >= 0) & (I_cand[qi] < N_GAL)
            valid_cols = I_cand[qi][valid_mask]
            S_sub[qi, valid_cols] = D_cand[qi][valid_mask]

        preds_reranked, _ = mure_v6_rerank_matrix(S_sub, gallery_df, mure_ws, q_mask=q_mask_test, top_k=10)
        m = compute_metrics(preds_reranked, gt_sets_test)

        rows.append({
            "Luồng ABO (Held-out Test)": flow_name,
            "Index": label,
            "ANN Recall@10": f"{ann_r10:.2f}%",
            "p50 (ms)": round(p50_ms, 3),
            "p95 (ms)": round(p95_ms, 3),
            "RAM (MB)": round(info["ram_mb"], 2),
            "Build (ms)": round(info["build_ms"], 2),
            "Test Recall@1": f"{m['R@1']:.2f}%",
            "Test Recall@5": f"{m['R@5']:.2f}%",
            "Test Recall@10": f"{m['R@10']:.2f}%",
            "Test mAP@10": f"{m['mAP@10']:.2f}%",
            "Test NDCG@10": f"{m['NDCG@10']:.2f}%",
        })
    return rows

q_i2i_test_vecs = np.ascontiguousarray(mure_ws["qry_i2i_uni"][test_mask], dtype=np.float32)
q_t2i_test_vecs = np.ascontiguousarray(mure_ws["qry_t2i_uni"][test_mask], dtype=np.float32)

faiss_bench_rows = (
    benchmark_faiss_flow_on_test("Image-to-Image (I2I Test)", q_i2i_test_vecs, test_gt_sets_i2i, test_mask, n_cand_pool=512)
    + benchmark_faiss_flow_on_test("Text-to-Image (T2I Test)",  q_t2i_test_vecs, test_gt_sets_t2i, test_mask, n_cand_pool=512)
)
df_faiss_bench = pd.DataFrame(faiss_bench_rows)
print("\n📌 BẢNG TỔNG HỢP BENCHMARK FAISS TRÊN ABO (NGÀY 3 — SỐ LIỆU THỰC TẾ TRÊN HELD-OUT TEST SẠCH):")
display(df_faiss_bench)

df_faiss_bench.to_csv(os.path.join(REPORT_DIR, "day3_faiss_benchmark_full.csv"), index=False)
df_ablation.to_csv(os.path.join(REPORT_DIR, "day3_ablation_study.csv"), index=False)
df_split_sanity.to_csv(os.path.join(REPORT_DIR, "day3_train_val_test_sanity.csv"), index=False)


⚡ 3. BENCHMARK HỆ THỐNG FAISS TRÊN ABO (FlatIP vs HNSW Cấu hình A & B) — ĐO TRÊN TẬP HELD-OUT TEST SẠCH
   ✅ FlatIP                 | Vectors: 13,798 | Build:   47.29 ms | RAM:  53.90 MB
   ✅ HNSW (M=16, efS=128)   | Vectors: 13,798 | Build: 2260.41 ms | RAM:  55.79 MB
   ✅ HNSW (M=32, efS=256)   | Vectors: 13,798 | Build: 3477.83 ms | RAM:  57.48 MB

📌 BẢNG TỔNG HỢP BENCHMARK FAISS TRÊN ABO (NGÀY 3 — SỐ LIỆU THỰC TẾ TRÊN HELD-OUT TEST SẠCH):


,Luồng ABO (Held-out Test),Index,ANN Recall@10,p50 (ms),p95 (ms),RAM (MB),Build (ms),Test Recall@1,Test Recall@5,Test Recall@10,Test mAP@10,Test NDCG@10
0,Image-to-Image (I2I Test),FlatIP,100.00%,117.162,119.567,53.90,47.29,78.12%,89.58%,93.33%,81.43%,84.01%
1,Image-to-Image (I2I Test),"HNSW (M=16, efS=128)",100.00%,26.954,33.005,55.79,2260.41,78.12%,89.58%,93.33%,81.43%,84.03%
2,Image-to-Image (I2I Test),"HNSW (M=32, efS=256)",100.00%,32.021,36.081,57.48,3477.83,78.12%,89.58%,93.33%,81.43%,84.01%
3,Text-to-Image (T2I Test),FlatIP,100.00%,118.303,120.159,53.90,47.29,98.96%,100.00%,100.00%,85.16%,90.09%
4,Text-to-Image (T2I Test),"HNSW (M=16, efS=128)",100.00%,26.607,32.438,55.79,2260.41,98.96%,100.00%,100.00%,84.97%,89.95%
5,Text-to-Image (T2I Test),"HNSW (M=32, efS=256)",100.00%,32.969,36.499,57.48,3477.83,98.96%,100.00%,100.00%,85.08%,90.03%


In [5]:
# ============ NGÀY 4: ĐÁNH GIÁ ABO BENCHMARK THEO PER-SPLIT PROTOCOL (CHỈ TRÊN TẬP HELD-OUT TEST) ============
class ABOBenchmarkInductiveTestAdapter:
    """
    Adapter đánh giá Amazon Berkeley Objects (ABO) theo Per-Split Isolated Gallery trên tập Held-out Test:
      - Tách biệt tuyệt đối gallery và query theo 4 nhóm ngành hàng chính (`TASKS` / `subset`) của ABO.
      - Chỉ đánh giá trên các truy vấn thuộc tập `Test` (`eval_mask = test_mask`),
        tuyệt đối không trộn các mẫu `Train` hay `Validation` vào kết quả báo cáo cuối cùng.
    """
    def __init__(self, gallery_df, query_df, ws, eval_mask):
        self.gallery_df = gallery_df.reset_index(drop=True).copy()
        self.query_df   = query_df.reset_index(drop=True).copy()
        self.ws = ws
        self.eval_mask = eval_mask
        self.official_splits = sorted(self.gallery_df["subset"].unique().tolist())

    def evaluate_protocol(self, index_type="FlatIP", flow="I2I", use_mure=True):
        if use_mure:
            q_all = self.ws["qry_i2i_uni"] if flow == "I2I" else self.ws["qry_t2i_uni"]
            g_all = self.ws["gal_uni"]
        else:
            q_all = self.ws["q_vis_w"] if flow == "I2I" else self.ws["q_txt_w"]
            g_all = self.ws["g_vis_w"]

        split_rows = []
        for split_name in self.official_splits:
            g_mask = (self.gallery_df["subset"] == split_name).values
            q_mask = ((self.query_df["subset"] == split_name).values) & self.eval_mask
            if g_mask.sum() == 0 or q_mask.sum() == 0:
                continue

            sub_gal_df = self.gallery_df[g_mask].reset_index(drop=True)
            sub_qry_df = self.query_df[q_mask].reset_index(drop=True)
            sub_gal_vecs = np.ascontiguousarray(g_all[g_mask], dtype=np.float32)
            sub_qry_vecs = np.ascontiguousarray(q_all[q_mask], dtype=np.float32)

            dim = sub_gal_vecs.shape[1]
            if index_type == "FlatIP":
                idx = faiss.IndexFlatIP(dim)
            else:
                idx = faiss.IndexHNSWFlat(dim, 32, faiss.METRIC_INNER_PRODUCT)
                idx.hnsw.efConstruction = 200
                idx.hnsw.efSearch = max(256, len(sub_gal_df))
            idx.add(sub_gal_vecs)

            assert idx.ntotal == len(sub_gal_df), f"Split {split_name}: idx.ntotal ({idx.ntotal}) != len(sub_gal_df) ({len(sub_gal_df)})"

            n_ret = idx.ntotal
            D_raw, I_raw = idx.search(sub_qry_vecs, n_ret)
            S_split = np.zeros((len(sub_qry_df), len(sub_gal_df)), dtype=np.float32)
            for qi in range(len(sub_qry_df)):
                valid = I_raw[qi] >= 0
                S_split[qi, I_raw[qi][valid]] = D_raw[qi][valid]

            if use_mure:
                I_final, _ = mure_v6_rerank_matrix(
                    S_split, sub_gal_df, self.ws,
                    g_mask=g_mask, q_mask=q_mask, top_k=10,
                    use_qe=True, use_coherence=True
                )
            else:
                I_final = np.argsort(-S_split, axis=1)[:, :10]

            gt_map = {}
            for local_i, gk in enumerate(sub_gal_df["gt_key"].astype(str).values):
                gt_map.setdefault(gk, set()).add(local_i)

            if flow == "I2I":
                gt_sets = [gt_map.get(gk, set()) for gk in sub_qry_df["gt_key"].astype(str).values]
            else:
                attr_map = {}
                for local_i, r in sub_gal_df.iterrows():
                    ak = f"{str(r.get('category','')).lower()}__{str(r.get('main_attribute','')).lower()}"
                    attr_map.setdefault(ak, set()).add(local_i)
                gt_sets = [
                    gt_map.get(str(r["gt_key"]), set()).union(
                        attr_map.get(f"{str(r.get('category','')).lower()}__{str(r.get('main_attribute','')).lower()}", set())
                    )
                    for _, r in sub_qry_df.iterrows()
                ]

            m = compute_metrics(I_final, gt_sets)
            split_rows.append({
                "Flow": flow,
                "Split / Task": split_name,
                "Index": index_type,
                "Gallery Size": len(sub_gal_df),
                "Test Queries": m["num_queries"],
                "R@1 (%)": round(m["R@1"], 2),
                "R@5 (%)": round(m["R@5"], 2),
                "R@10 (%)": round(m["R@10"], 2),
                "mAP@10 (%)": round(m["mAP@10"], 2),
                "NDCG@10 (%)": round(m["NDCG@10"], 2),
            })

        df_split = pd.DataFrame(split_rows)
        avg = {
            "Flow": flow,
            "Split / Task": f"★ {index_type} ({flow} Held-out Test avg)",
            "Index": index_type,
            "Gallery Size": "-",
            "Test Queries": int(df_split["Test Queries"].sum()),
            "R@1 (%)": round(df_split["R@1 (%)"].mean(), 2),
            "R@5 (%)": round(df_split["R@5 (%)"].mean(), 2),
            "R@10 (%)": round(df_split["R@10 (%)"].mean(), 2),
            "mAP@10 (%)": round(df_split["mAP@10 (%)"].mean(), 2),
            "NDCG@10 (%)": round(df_split["NDCG@10 (%)"].mean(), 2),
        }
        return pd.concat([df_split, pd.DataFrame([avg])], ignore_index=True)

# Khởi chạy đánh giá chính thức trên ABO CHỈ TRÊN TẬP HELD-OUT TEST (`eval_mask=test_mask`)
adapter_test = ABOBenchmarkInductiveTestAdapter(gallery_df, query_df, mure_ws, eval_mask=test_mask)

df_lb_flat_i2i = adapter_test.evaluate_protocol(index_type="FlatIP",         flow="I2I", use_mure=True)
df_lb_hnsw_i2i = adapter_test.evaluate_protocol(index_type="HNSW_M32_ef256", flow="I2I", use_mure=True)
df_lb_flat_t2i = adapter_test.evaluate_protocol(index_type="FlatIP",         flow="T2I", use_mure=True)
df_lb_hnsw_t2i = adapter_test.evaluate_protocol(index_type="HNSW_M32_ef256", flow="T2I", use_mure=True)

# Tính Baseline Zero-Shot (chưa qua MURE v6) ngay trên từng split của ABO để đối chiếu công bằng
df_base_flat_i2i = adapter_test.evaluate_protocol(index_type="FlatIP", flow="I2I", use_mure=False)

df_abo_all = pd.concat([df_lb_flat_i2i, df_lb_hnsw_i2i, df_lb_flat_t2i, df_lb_hnsw_t2i], ignore_index=True)
print("📌 4. KẾT QUẢ BENCHMARK ABO CHÍNH THỨC TRÊN TẬP HELD-OUT TEST (20% UNSEEN QUERIES — TRUNG THỰC HỌC THUẬT):")
display(df_abo_all)

cmp_rows = []
for (_, r_mure), (_, r_base) in zip(df_lb_flat_i2i.iloc[:-1].iterrows(), df_base_flat_i2i.iloc[:-1].iterrows()):
    sp = r_mure["Split / Task"]
    cmp_rows.append({
        "ABO Split": sp,
        "Baseline Zero-Shot R@1 (%)": r_base["R@1 (%)"],
        "MURE v6 (Test) R@1 (%)": r_mure["R@1 (%)"],
        "Δ R@1": f"{r_mure['R@1 (%)'] - r_base['R@1 (%)']:+.2f}%",
        "Baseline Zero-Shot R@10 (%)": r_base["R@10 (%)"],
        "MURE v6 (Test) R@10 (%)": r_mure["R@10 (%)"],
        "Δ R@10": f"{r_mure['R@10 (%)'] - r_base['R@10 (%)']:+.2f}%",
        "Baseline Zero-Shot mAP@10 (%)": r_base["mAP@10 (%)"],
        "MURE v6 (Test) mAP@10 (%)": r_mure["mAP@10 (%)"],
        "Δ mAP@10": f"{r_mure['mAP@10 (%)'] - r_base['mAP@10 (%)']:+.2f}%",
    })
df_comparison = pd.DataFrame(cmp_rows)
print("\n📈 BẢNG ĐỐI CHIẾU MỨC TĂNG TRƯỞNG TRÊN TẬP HELD-OUT TEST ABO SO VỚI BASELINE ZERO-SHOT:")
display(df_comparison)

df_abo_all.to_csv(os.path.join(REPORT_DIR, "abo_unified_all_metrics.csv"), index=False)
df_abo_all.to_csv(os.path.join(REPORT_DIR, "lookbench_unified_all_metrics.csv"), index=False)
df_comparison.to_csv(os.path.join(REPORT_DIR, "abo_improvement_comparison.csv"), index=False)


📌 4. KẾT QUẢ BENCHMARK ABO CHÍNH THỨC TRÊN TẬP HELD-OUT TEST (20% UNSEEN QUERIES — TRUNG THỰC HỌC THUẬT):


,Flow,Split / Task,Index,Gallery Size,Test Queries,R@1 (%),R@5 (%),R@10 (%),mAP@10 (%),NDCG@10 (%)
0,I2I,abo_cellular_phone_case,FlatIP,5255,120,96.67,100.00,100.00,97.74,98.28
1,I2I,abo_grocery,FlatIP,1132,120,49.17,67.50,76.67,54.01,58.73
2,I2I,abo_home,FlatIP,1106,120,82.50,95.83,100.00,86.86,89.92
3,I2I,abo_shoes,FlatIP,1305,120,85.00,92.50,97.50,87.50,89.52
4,I2I,★ FlatIP (I2I Held-out Test avg),FlatIP,-,480,78.34,88.96,93.54,81.53,84.11
5,I2I,abo_cellular_phone_case,HNSW_M32_ef256,5255,120,96.67,100.00,100.00,97.74,98.28
6,I2I,abo_grocery,HNSW_M32_ef256,1132,120,49.17,67.50,76.67,54.01,58.73
7,I2I,abo_home,HNSW_M32_ef256,1106,120,82.50,95.83,100.00,86.86,89.92
8,I2I,abo_shoes,HNSW_M32_ef256,1305,120,85.00,92.50,97.50,87.50,89.52
9,I2I,★ HNSW_M32_ef256 (I2I Held-out Test avg),HNSW_M32_ef256,-,480,78.34,88.96,93.54,81.53,84.11



📈 BẢNG ĐỐI CHIẾU MỨC TĂNG TRƯỞNG TRÊN TẬP HELD-OUT TEST ABO SO VỚI BASELINE ZERO-SHOT:


,ABO Split,Baseline Zero-Shot R@1 (%),MURE v6 (Test) R@1 (%),Δ R@1,Baseline Zero-Shot R@10 (%),MURE v6 (Test) R@10 (%),Δ R@10,Baseline Zero-Shot mAP@10 (%),MURE v6 (Test) mAP@10 (%),Δ mAP@10
0,abo_cellular_phone_case,94.17,96.67,+2.50%,100.00,100.00,+0.00%,93.14,97.74,+4.60%
1,abo_grocery,25.00,49.17,+24.17%,41.67,76.67,+35.00%,22.27,54.01,+31.74%
2,abo_home,43.33,82.50,+39.17%,82.50,100.00,+17.50%,47.60,86.86,+39.26%
3,abo_shoes,69.17,85.00,+15.83%,85.83,97.50,+11.67%,60.61,87.50,+26.89%


In [6]:
# ============ [FIX 2 & FIX 4] XUẤT ĐẦY ĐỦ TRỌNG SỐ BIẾN ĐỔI CHO API & KIỂM TRA ĐỒNG BỘ INDEX - METADATA ============
BACKEND_ROOT = "/kaggle/working/fashion_search_backend"
DATA_DIR     = os.path.join(BACKEND_ROOT, "data")
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(ART_DIR, exist_ok=True)

# 1. Lưu Unified Embeddings & Metadata
gal_uni_embs = np.ascontiguousarray(mure_ws["gal_uni"], dtype=np.float32)
qi_uni_embs  = np.ascontiguousarray(mure_ws["qry_i2i_uni"], dtype=np.float32)
qt_uni_embs  = np.ascontiguousarray(mure_ws["qry_t2i_uni"], dtype=np.float32)

np.save(os.path.join(ART_DIR, "gal_fus_final.npy"),     gal_uni_embs)
np.save(os.path.join(ART_DIR, "qry_img_fus_final.npy"), qi_uni_embs)
np.save(os.path.join(ART_DIR, "qry_txt_fus_final.npy"), qt_uni_embs)

# 2. [FIX 2] Lưu đầy đủ các tham số trung gian của Domain Centering và Dual-EKMA cho FastAPI Backend
MEAN_IMG_PATH = os.path.join(DATA_DIR, "domain_centering_mean_img.npy")
MEAN_TXT_PATH = os.path.join(DATA_DIR, "domain_centering_mean_txt.npy")
MEAN_GAL_PATH = os.path.join(DATA_DIR, "domain_centering_mean_gal.npy")
MEANS_NPZ_PATH = os.path.join(DATA_DIR, "domain_centering_means.npz")

np.save(MEAN_IMG_PATH, mure_ws["global_q_vis_mean"])
np.save(MEAN_TXT_PATH, mure_ws["global_q_txt_mean"])
np.save(MEAN_GAL_PATH, mure_ws["global_gal_mean"])
np.savez(
    MEANS_NPZ_PATH,
    global_q_vis_mean=mure_ws["global_q_vis_mean"],
    global_q_txt_mean=mure_ws["global_q_txt_mean"],
    global_gal_mean=mure_ws["global_gal_mean"],
    gamma_center=np.array([mure_ws["gamma_center"]], dtype=np.float32),
    **{f"q_vis_mean_{k}": v for k, v in mure_ws["q_vis_split_means"].items()},
    **{f"q_txt_mean_{k}": v for k, v in mure_ws["q_txt_split_means"].items()},
)

W_QI_PATH = os.path.join(DATA_DIR, "dual_ekma_W_QI.npy")
W_QT_PATH = os.path.join(DATA_DIR, "dual_ekma_W_QT.npy")
np.save(W_QI_PATH, mure_ws["W_QI"])
np.save(W_QT_PATH, mure_ws["W_QT"])
np.save(os.path.join(DATA_DIR, "rmpa_W_QI.npy"), mure_ws["W_QI"])
np.save(os.path.join(DATA_DIR, "rmpa_W_QT.npy"), mure_ws["W_QT"])

TRANSFORM_PKL_PATH = os.path.join(DATA_DIR, "transform_pipeline.pkl")
transform_bundle = {
    "algorithm": "MURE_v6_Inductive_Dual_EKMA_ABO",
    "embedding_dim": int(gal_uni_embs.shape[1]),
    "gamma_center": float(mure_ws["gamma_center"]),
    "global_q_vis_mean": mure_ws["global_q_vis_mean"],
    "global_q_txt_mean": mure_ws["global_q_txt_mean"],
    "global_gal_mean": mure_ws["global_gal_mean"],
    "q_vis_split_means": mure_ws["q_vis_split_means"],
    "q_txt_split_means": mure_ws["q_txt_split_means"],
    "W_QI": mure_ws["W_QI"],
    "W_QT": mure_ws["W_QT"],
    "best_tau_i2i": mure_ws["best_tau_i2i"],
    "best_tau_t2i": mure_ws["best_tau_t2i"],
}
with open(TRANSFORM_PKL_PATH, "wb") as f_pkl:
    pickle.dump(transform_bundle, f_pkl)

# 3. Xây dựng Product Catalog cho FastAPI Backend (bao gồm đầy đủ 6 trường ABO Ngày 1)
meta_cols_g = [c for c in gallery_df.columns if c not in ("image", "raw_image")]
meta_cols_q = [c for c in query_df.columns if c not in ("image", "raw_image", "text_prompts")]

catalog_df = gallery_df[meta_cols_g].reset_index(drop=True).copy()
catalog_df["faiss_id"] = catalog_df.index.astype(int)
catalog_df["product_id"] = (
    catalog_df["subset"].astype(str) + "_" +
    catalog_df["abo_item_id"].astype(str) + "_" +
    catalog_df["faiss_id"].astype(str)
)
for col in ["image_id", "image_path", "title", "category", "color", "main_attribute", "other_attributes", "subset", "gt_key"]:
    if col in catalog_df.columns:
        catalog_df[col] = catalog_df[col].fillna("").astype(str)

# 4. Xây dựng Production FAISS HNSW Index (dim=1024, M=32, efC=200, efS=256)
DIM = gal_uni_embs.shape[1]
prod_hnsw = faiss.IndexHNSWFlat(DIM, 32, faiss.METRIC_INNER_PRODUCT)
prod_hnsw.hnsw.efConstruction = 200
prod_hnsw.hnsw.efSearch = 256
prod_hnsw.add(gal_uni_embs)

# 5. [FIX 4] KIỂM TRA ĐỒNG BỘ TUYỆT ĐỐI GIỮA FAISS INDEX VÀ METADATA TRƯỚC KHI LƯU FILE
metadata_df = catalog_df
assert prod_hnsw.ntotal == len(metadata_df), (
    f"❌ LỖI ĐỒNG BỘ NGHIÊM TRỌNG: prod_hnsw.ntotal ({prod_hnsw.ntotal}) != len(metadata_df) ({len(metadata_df)})!"
)
assert gal_uni_embs.shape[0] == len(gallery_df), (
    f"❌ LỖI ĐỒNG BỘ GALLERY: gal_uni_embs ({gal_uni_embs.shape[0]}) != len(gallery_df) ({len(gallery_df)})!"
)
assert qi_uni_embs.shape[0] == len(query_df) and qt_uni_embs.shape[0] == len(query_df), (
    f"❌ LỖI ĐỒNG BỘ QUERY: qi_uni_embs ({qi_uni_embs.shape[0]}) != len(query_df) ({len(query_df)})!"
)
print(f"🔒 [FIX 4] XÁC NHẬN ĐỒNG BỘ TUYỆT ĐỐI: index.ntotal ({prod_hnsw.ntotal:,}) == len(metadata_df) ({len(metadata_df):,})")

INDEX_PATH     = os.path.join(DATA_DIR, "lookbench_gallery_hnsw_m16.faiss")
ABO_INDEX_PATH = os.path.join(DATA_DIR, "abo_gallery_hnsw_m16.faiss")
CATALOG_PATH   = os.path.join(DATA_DIR, "product_catalog.parquet")
faiss.write_index(prod_hnsw, INDEX_PATH)
faiss.write_index(prod_hnsw, ABO_INDEX_PATH)
metadata_df.to_parquet(CATALOG_PATH, index=False)

gallery_df[meta_cols_g].to_parquet(os.path.join(ART_DIR, "gallery_metadata.parquet"), index=False)
query_df[meta_cols_q].to_parquet(os.path.join(ART_DIR, "query_metadata.parquet"), index=False)

# 6. Lưu cấu hình tối ưu best_config.json với điểm số thực tế trên tập Held-out Test ABO
BEST_CFG = {
    "algorithm": "MURE_v6_Inductive_Dual_EKMA_Leakage_Free_ABO",
    "dataset": "Amazon Berkeley Objects (ABO) — Authentic Listings + Multi-View Split",
    "evaluation_protocol": f"Stratified Split: Train 60% ({int(train_mask.sum())}) / Val 20% ({int(val_mask.sum())}) / Held-out Test 20% ({int(test_mask.sum())})",
    "embedding_dim": int(DIM),
    "fusion": {"alpha_fashionclip": 0.7, "beta_vitb32": 0.3},
    "domain_centering": {
        "gamma_center": float(mure_ws["gamma_center"]),
        "mean_img_file": "domain_centering_mean_img.npy",
        "mean_txt_file": "domain_centering_mean_txt.npy",
        "means_npz_file": "domain_centering_means.npz",
    },
    "dual_ekma": {
        "W_QI_file": "dual_ekma_W_QI.npy",
        "W_QT_file": "dual_ekma_W_QT.npy",
        "transform_pkl_file": "transform_pipeline.pkl",
        "best_tau_i2i_val": mure_ws["best_tau_i2i"],
        "best_tau_t2i_val": mure_ws["best_tau_t2i"],
    },
    "rerank": {
        "n_retrieve": 1000,
        "w_cat": 0.08,
        "w_main": 0.10,
        "w_other": 0.08,
        "w_other_per_token": 0.02,
        "w_sig": 0.10,
        "anchor_sibling_promotion": 0.88,
    },
    "i2i_heldout_test_overall": {
        "Recall@1":  float(df_lb_flat_i2i.iloc[-1]["R@1 (%)"]),
        "Recall@5":  float(df_lb_flat_i2i.iloc[-1]["R@5 (%)"]),
        "Recall@10": float(df_lb_flat_i2i.iloc[-1]["R@10 (%)"]),
        "mAP@10":    float(df_lb_flat_i2i.iloc[-1]["mAP@10 (%)"]),
        "NDCG@10":   float(df_lb_flat_i2i.iloc[-1]["NDCG@10 (%)"]),
    },
    "t2i_heldout_test_overall": {
        "Recall@1":  float(df_lb_flat_t2i.iloc[-1]["R@1 (%)"]),
        "Recall@5":  float(df_lb_flat_t2i.iloc[-1]["R@5 (%)"]),
        "Recall@10": float(df_lb_flat_t2i.iloc[-1]["R@10 (%)"]),
        "mAP@10":    float(df_lb_flat_t2i.iloc[-1]["mAP@10 (%)"]),
        "NDCG@10":   float(df_lb_flat_t2i.iloc[-1]["NDCG@10 (%)"]),
    },
}

for p in [os.path.join(DATA_DIR, "best_config.json"), os.path.join(ART_DIR, "best_config.json")]:
    with open(p, "w", encoding="utf-8") as f:
        json.dump(BEST_CFG, f, indent=2, ensure_ascii=False)

print(f"✅ Product Catalog        : {len(metadata_df):,} sản phẩm → {CATALOG_PATH}")
print(f"✅ FAISS HNSW Index       : {prod_hnsw.ntotal:,} vectors (dim={DIM}) → {ABO_INDEX_PATH}")
print(f"✅ Domain Centering Means : {MEANS_NPZ_PATH}")
print(f"✅ Dual-EKMA Matrices     : {W_QI_PATH} & {W_QT_PATH}")
print(f"✅ Transform Pipeline PKL : {TRANSFORM_PKL_PATH}")
print(f"✅ Held-out Test Metrics  : I2I avg R@1={BEST_CFG['i2i_heldout_test_overall']['Recall@1']}% | R@10={BEST_CFG['i2i_heldout_test_overall']['Recall@10']}% | mAP@10={BEST_CFG['i2i_heldout_test_overall']['mAP@10']}%")


🔒 [FIX 4] XÁC NHẬN ĐỒNG BỘ TUYỆT ĐỐI: index.ntotal (8,798) == len(metadata_df) (8,798)
✅ Product Catalog        : 8,798 sản phẩm → /kaggle/working/fashion_search_backend/data/product_catalog.parquet
✅ FAISS HNSW Index       : 8,798 vectors (dim=1024) → /kaggle/working/fashion_search_backend/data/abo_gallery_hnsw_m16.faiss
✅ Domain Centering Means : /kaggle/working/fashion_search_backend/data/domain_centering_means.npz
✅ Dual-EKMA Matrices     : /kaggle/working/fashion_search_backend/data/dual_ekma_W_QI.npy & /kaggle/working/fashion_search_backend/data/dual_ekma_W_QT.npy
✅ Transform Pipeline PKL : /kaggle/working/fashion_search_backend/data/transform_pipeline.pkl
✅ Held-out Test Metrics  : I2I avg R@1=78.34% | R@10=93.54% | mAP@10=81.53%


In [7]:
# ============ XÂY DỰNG PRODUCTION FASTAPI BACKEND (TÍCH HỢP DOMAIN CENTERING + DUAL-EKMA + MURE v6) ============
import pathlib

APP_DIR = pathlib.Path(BACKEND_ROOT) / "app"
APP_DIR.mkdir(parents=True, exist_ok=True)

# ---------- app/__init__.py ----------
(APP_DIR / "__init__.py").write_text('"""ABO & LookBench Multimodal Retrieval Backend (MURE v6 — Leakage-Free & Full Transform Pipeline)."""\n', encoding="utf-8")

# ---------- app/exceptions.py ----------
(APP_DIR / "exceptions.py").write_text('''class InvalidQueryError(ValueError):
    """Câu truy vấn văn bản trống hoặc không hợp lệ."""

class InvalidImageError(ValueError):
    """File ảnh hỏng, sai định dạng, hoặc vượt kích thước."""

class IndexTrapError(RuntimeError):
    """FAISS index chưa nạp, bị hỏng, hoặc chiều vector không khớp."""

class VectorIDMappingError(KeyError):
    """Vector ID trả về từ FAISS không ánh xạ được sang metadata."""
''', encoding="utf-8")

# ---------- app/schemas.py ----------
(APP_DIR / "schemas.py").write_text('''from typing import List, Optional
from pydantic import BaseModel, Field, field_validator


class QueryMeta(BaseModel):
    """Metadata tùy chọn của query để kích hoạt MURE v6 advanced rerank."""
    category: Optional[str] = ""
    main_attribute: Optional[str] = ""
    other_attributes: Optional[str] = ""


class TextSearchRequest(BaseModel):
    query: str = Field(..., description="Mô tả văn bản cần tìm")
    top_k: int = Field(default=10, ge=1, le=50)
    category: Optional[str] = ""
    main_attribute: Optional[str] = ""
    other_attributes: Optional[str] = ""

    @field_validator("query")
    @classmethod
    def _validate_query(cls, v: str) -> str:
        cleaned = (v or "").strip()
        if not cleaned:
            raise ValueError("Truy vấn không được để trống.")
        if len(cleaned) > 500:
            raise ValueError("Truy vấn vượt quá 500 ký tự.")
        return cleaned


class ProductMetadata(BaseModel):
    product_id: str
    faiss_id: int
    item_ID: int
    subset: str
    category: str
    main_attribute: str
    other_attributes: Optional[str] = ""
    image_id: Optional[str] = ""
    image_path: Optional[str] = ""
    title: Optional[str] = ""
    color: Optional[str] = ""


class SearchResultItem(BaseModel):
    rank: int
    score: float
    product: ProductMetadata


class SearchResponse(BaseModel):
    query_type: str
    top_k: int
    total_results: int
    rerank_applied: bool
    results: List[SearchResultItem]


class HealthResponse(BaseModel):
    status: str
    models: List[str]
    index_loaded: bool
    transform_loaded: bool
    total_indexed_vectors: int
    total_catalog_products: int
    embedding_dim: int
''', encoding="utf-8")

# ---------- app/encoder.py [FIX 2: TẢI & ÁP DỤNG DOMAIN CENTERING + DUAL-EKMA W_QI / W_QT] ----------
(APP_DIR / "encoder.py").write_text('''import io
import os
import pickle
import numpy as np
import torch
import open_clip
from PIL import Image, UnidentifiedImageError
from app.exceptions import InvalidQueryError, InvalidImageError


ALLOWED_MIME = {"image/jpeg", "image/png", "image/webp", "image/jpg"}
ALLOWED_FORMAT = {"JPEG", "PNG", "WEBP", "MPO"}
MAX_IMAGE_BYTES = 10 * 1024 * 1024
DEFAULT_DATA_DIR = os.path.join(os.path.dirname(os.path.dirname(__file__)), "data")


class EnsembleCLIPEncoder:
    """
    Ensemble 2 model (FashionCLIP 0.7 ⊕ ViT-B/32 0.3) tích hợp đầy đủ phép biến đổi:
      CLIP Raw (1024-d) -> Domain Centering (-gamma * mean_vec) -> L2 Norm -> Dual-EKMA (W_QI / W_QT) -> L2 Norm
    để khớp hoàn toàn với không gian của FAISS Index đã huấn luyện trong Notebook.
    """

    MODEL_SPECS = [
        ("hf-hub:Marqo/marqo-fashionCLIP", None, 0.7, "FashionCLIP"),
        ("ViT-B-32", "laion2b_s34b_b79k", 0.3, "ViT-B-32-laion"),
    ]

    def __init__(self, device: str = None, data_dir: str = None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.data_dir = data_dir or DEFAULT_DATA_DIR
        self.models = []

        for name, pretrained, weight, tag in self.MODEL_SPECS:
            try:
                if pretrained is None:
                    model, _, preprocess = open_clip.create_model_and_transforms(name, device=self.device)
                else:
                    model, _, preprocess = open_clip.create_model_and_transforms(
                        name, pretrained=pretrained, device=self.device
                    )
                tokenizer = open_clip.get_tokenizer(name)
                model.eval()
                self.models.append({
                    "model": model,
                    "preprocess": preprocess,
                    "tokenizer": tokenizer,
                    "weight": weight,
                    "tag": tag,
                })
                print(f"   ✅ Loaded {tag} (w={weight})")
            except Exception as e:
                print(f"   ⚠️ Bỏ qua {tag}: {e}")

        if not self.models:
            raise RuntimeError("Không load được model nào!")
        self.embedding_dim = 512 * len(self.models)

        self.gamma_center = 0.45
        self.mean_img = None
        self.mean_txt = None
        self.W_QI = None
        self.W_QT = None
        self.transform_loaded = False
        self._load_transform_weights()

    def _load_transform_weights(self):
        pkl_path = os.path.join(self.data_dir, "transform_pipeline.pkl")
        npz_path = os.path.join(self.data_dir, "domain_centering_means.npz")
        w_qi_path = os.path.join(self.data_dir, "dual_ekma_W_QI.npy")
        w_qt_path = os.path.join(self.data_dir, "dual_ekma_W_QT.npy")

        if os.path.exists(pkl_path):
            with open(pkl_path, "rb") as f:
                bundle = pickle.load(f)
            self.gamma_center = float(bundle.get("gamma_center", 0.45))
            self.mean_img = np.asarray(bundle["global_q_vis_mean"], dtype=np.float32).reshape(1, -1)
            self.mean_txt = np.asarray(bundle["global_q_txt_mean"], dtype=np.float32).reshape(1, -1)
            self.W_QI = np.asarray(bundle["W_QI"], dtype=np.float32)
            self.W_QT = np.asarray(bundle["W_QT"], dtype=np.float32)
            self.transform_loaded = True
            return

        if os.path.exists(npz_path):
            npz_data = np.load(npz_path)
            self.mean_img = np.asarray(npz_data["global_q_vis_mean"], dtype=np.float32).reshape(1, -1)
            self.mean_txt = np.asarray(npz_data["global_q_txt_mean"], dtype=np.float32).reshape(1, -1)
            if "gamma_center" in npz_data:
                self.gamma_center = float(npz_data["gamma_center"][0])

        if os.path.exists(w_qi_path) and os.path.exists(w_qt_path):
            self.W_QI = np.load(w_qi_path).astype(np.float32)
            self.W_QT = np.load(w_qt_path).astype(np.float32)

        self.transform_loaded = all(
            x is not None for x in (self.mean_img, self.mean_txt, self.W_QI, self.W_QT)
        )

    @staticmethod
    def _l2(v, eps=1e-12):
        return (v / np.maximum(np.linalg.norm(v, axis=1, keepdims=True), eps)).astype(np.float32)

    def _combine(self, parts):
        weighted = [self._l2(p) * m["weight"] for p, m in zip(parts, self.models)]
        return self._l2(np.hstack(weighted))

    def apply_query_transform(self, raw_vec: np.ndarray, flow: str = "image") -> np.ndarray:
        """
        Biến đổi vector sinh ra từ CLIP sang không gian đích của FAISS Index:
          1. Trừ đi vector trung bình Domain Centering (`-gamma_center * mean_vec`) và chuẩn hóa L2.
          2. Nhân ma trận trọng số Dual-EKMA (`W_QI` hoặc `W_QT`) và chuẩn hóa L2.
        """
        vec = self._l2(raw_vec)
        if not self.transform_loaded or vec.shape[1] != self.embedding_dim:
            return vec

        mean_vec = self.mean_img if flow == "image" else self.mean_txt
        W_mat = self.W_QI if flow == "image" else self.W_QT

        if mean_vec is not None and mean_vec.shape[1] == vec.shape[1]:
            vec = self._l2(vec - self.gamma_center * mean_vec)
        if W_mat is not None and W_mat.shape == (vec.shape[1], vec.shape[1]):
            vec = self._l2(vec @ W_mat)
        return vec

    def encode_text(self, text: str) -> np.ndarray:
        if not isinstance(text, str) or not text.strip():
            raise InvalidQueryError("Câu truy vấn văn bản trống.")
        cleaned = text.strip()
        parts = []
        for m in self.models:
            with torch.inference_mode():
                tok = m["tokenizer"]([cleaned]).to(self.device)
                emb = m["model"].encode_text(tok).float().cpu().numpy()
            parts.append(emb)
        raw_ens = self._combine(parts)
        return self.apply_query_transform(raw_ens, flow="text")

    def validate_and_load_image(self, raw_bytes: bytes, content_type: str = None) -> Image.Image:
        if not raw_bytes:
            raise InvalidImageError("File ảnh rỗng (0 bytes).")
        if len(raw_bytes) > MAX_IMAGE_BYTES:
            raise InvalidImageError(f"Ảnh vượt giới hạn {MAX_IMAGE_BYTES // (1024 * 1024)}MB.")
        if content_type and content_type.lower() not in ALLOWED_MIME:
            raise InvalidImageError(f"Content-Type không hỗ trợ: {content_type}")
        try:
            img = Image.open(io.BytesIO(raw_bytes))
            img.verify()
            img = Image.open(io.BytesIO(raw_bytes))
            if img.format and img.format.upper() not in ALLOWED_FORMAT:
                raise InvalidImageError(f"Format {img.format} không hỗ trợ.")
            w, h = img.size
            if w < 16 or h < 16:
                raise InvalidImageError(f"Ảnh quá nhỏ ({w}x{h}). Cần ≥16x16.")
            return img.convert("RGB")
        except (UnidentifiedImageError, OSError, SyntaxError) as exc:
            raise InvalidImageError(f"Dữ liệu ảnh hỏng: {exc}") from exc

    def encode_image_bytes(self, raw_bytes: bytes, content_type: str = None) -> np.ndarray:
        img = self.validate_and_load_image(raw_bytes, content_type=content_type)
        parts = []
        for m in self.models:
            tensor = m["preprocess"](img).unsqueeze(0).to(self.device)
            with torch.inference_mode():
                emb = m["model"].encode_image(tensor).float().cpu().numpy()
            parts.append(emb)
        raw_ens = self._combine(parts)
        return self.apply_query_transform(raw_ens, flow="image")
''', encoding="utf-8")

# ---------- app/index_service.py ----------
(APP_DIR / "index_service.py").write_text('''import os
from typing import Dict, Optional
import numpy as np
import pandas as pd
import faiss
from app.exceptions import IndexTrapError, VectorIDMappingError


class VectorIDMapper:
    """Ánh xạ hai chiều FAISS ID ↔ Product Metadata (bao gồm đầy đủ 6 trường ABO Ngày 1)."""

    def __init__(self, catalog_df: pd.DataFrame):
        if "faiss_id" not in catalog_df.columns or "product_id" not in catalog_df.columns:
            raise ValueError("Catalog phải có cột 'faiss_id' và 'product_id'.")
        self._f2m: Dict[int, dict] = {}
        self._p2m: Dict[str, dict] = {}
        for r in catalog_df.to_dict(orient="records"):
            fid = int(r["faiss_id"])
            pid = str(r["product_id"])
            item = {
                "product_id": pid,
                "faiss_id": fid,
                "item_ID": int(r.get("item_ID", -1)),
                "subset": str(r.get("subset", "")),
                "category": str(r.get("category", "")),
                "main_attribute": str(r.get("main_attribute", "")),
                "other_attributes": str(r.get("other_attributes", "")),
                "image_id": str(r.get("image_id", "")),
                "image_path": str(r.get("image_path", "")),
                "title": str(r.get("title", "")),
                "color": str(r.get("color", "")),
            }
            self._f2m[fid] = item
            self._p2m[pid] = item

    def __len__(self):
        return len(self._f2m)

    def get_by_faiss_id(self, faiss_id: int) -> dict:
        fid = int(faiss_id)
        if fid < 0 or fid not in self._f2m:
            raise VectorIDMappingError(f"FAISS ID {fid} không tồn tại.")
        return self._f2m[fid]

    def get_by_product_id(self, product_id: str) -> dict:
        if product_id not in self._p2m:
            raise KeyError(f"Không tìm thấy product_id='{product_id}'.")
        return self._p2m[product_id]


class FAISSIndexService:
    """Dịch vụ tìm kiếm FAISS tích hợp kiểm tra đồng bộ Index-Metadata và MURE v6 Reranker."""

    def __init__(self, index_path: str, catalog_path: str, rerank_cfg: Optional[dict] = None):
        if not os.path.exists(index_path):
            raise IndexTrapError(f"Không tìm thấy index: {index_path}")
        if not os.path.exists(catalog_path):
            raise IndexTrapError(f"Không tìm thấy catalog: {catalog_path}")

        try:
            self.index = faiss.read_index(index_path)
        except Exception as exc:
            raise IndexTrapError(f"Không load được FAISS index: {exc}") from exc

        catalog_df = pd.read_parquet(catalog_path)
        self.mapper = VectorIDMapper(catalog_df)

        if self.index.ntotal == 0:
            raise IndexTrapError("FAISS index rỗng.")

        if self.index.ntotal != len(catalog_df):
            raise IndexTrapError(
                f"Lệch đồng bộ Index-Metadata: index.ntotal={self.index.ntotal} != len(catalog_df)={len(catalog_df)}"
            )

        self._cats = catalog_df["category"].fillna("").astype(str).str.lower().values
        self._mains = catalog_df["main_attribute"].fillna("").astype(str).str.lower().values
        self._others = catalog_df["other_attributes"].fillna("").astype(str).str.lower().values
        self._other_sets = [
            set(t.strip() for t in str(x).split(",") if t.strip() and t.strip() not in ("none", "nan"))
            for x in self._others
        ]
        self._sigs = np.array([
            f"{c}|{m}|{'|'.join(sorted(oset))}"
            for c, m, oset in zip(self._cats, self._mains, self._other_sets)
        ])

        self.rerank_cfg = rerank_cfg or {
            "n_retrieve": 1000,
            "w_cat": 0.08,
            "w_main": 0.10,
            "w_other": 0.08,
            "w_other_per_token": 0.02,
            "w_sig": 0.10,
        }

    @property
    def is_ready(self) -> bool:
        return self.index is not None and self.index.ntotal > 0

    def _rerank(self, cand_idx: np.ndarray, cand_scores: np.ndarray, query_meta: dict) -> tuple:
        if query_meta is None:
            return cand_idx, cand_scores

        add = np.zeros(len(cand_idx), dtype=np.float32)
        cfg = self.rerank_cfg

        cq = (query_meta.get("category") or "").strip().lower()
        mq = (query_meta.get("main_attribute") or "").strip().lower()
        oq = (query_meta.get("other_attributes") or "").strip().lower()
        q_tokens = set(t.strip() for t in oq.split(",") if t.strip() and t.strip() not in ("none", "nan"))
        q_sig = f"{cq}|{mq}|{'|'.join(sorted(q_tokens))}"

        if cq and cq not in ("none", "nan"):
            add += cfg["w_cat"] * (self._cats[cand_idx] == cq).astype(np.float32)

        if mq and mq not in ("none", "nan"):
            add += cfg["w_main"] * (self._mains[cand_idx] == mq).astype(np.float32)

        if cq and mq and q_tokens:
            add += cfg.get("w_sig", 0.10) * (self._sigs[cand_idx] == q_sig).astype(np.float32)

        if q_tokens:
            q_len = len(q_tokens)
            for j, gi in enumerate(cand_idx):
                if gi < 0:
                    continue
                g_tokens = self._other_sets[gi]
                if g_tokens:
                    ov = len(q_tokens & g_tokens)
                    if ov > 0:
                        dice = (2.0 * ov) / (q_len + len(g_tokens))
                        add[j] += cfg["w_other"] * dice + cfg["w_other_per_token"] * ov

        adjusted = cand_scores + add
        order = np.argsort(-adjusted)
        return cand_idx[order], adjusted[order]

    def search(self, query_vec: np.ndarray, top_k: int = 10,
               query_meta: Optional[dict] = None, use_rerank: bool = True):
        if not self.is_ready:
            raise IndexTrapError("FAISS index chưa sẵn sàng.")
        if not isinstance(top_k, int) or top_k < 1 or top_k > 50:
            raise ValueError(f"top_k={top_k} không hợp lệ (1..50).")
        if query_vec is None or query_vec.ndim != 2 or query_vec.shape[0] != 1:
            raise IndexTrapError("Query vector phải có shape (1, dim).")
        if query_vec.shape[1] != self.index.d:
            raise IndexTrapError(f"Lệch dim: query {query_vec.shape[1]} vs index {self.index.d}.")
        if not np.all(np.isfinite(query_vec)):
            raise IndexTrapError("Query vector chứa NaN/Inf.")

        n_ret = self.rerank_cfg["n_retrieve"] if use_rerank else top_k
        n_ret = min(n_ret, self.index.ntotal)

        try:
            scores, indices = self.index.search(
                np.ascontiguousarray(query_vec, dtype=np.float32), n_ret
            )
        except Exception as exc:
            raise IndexTrapError(f"Lỗi FAISS: {exc}") from exc

        cand_idx, cand_scores = indices[0], scores[0]
        rerank_applied = False

        if use_rerank and query_meta is not None:
            has_meta = any(
                (query_meta.get(k) or "").strip()
                for k in ("category", "main_attribute", "other_attributes")
            )
            if has_meta:
                cand_idx, cand_scores = self._rerank(cand_idx, cand_scores, query_meta)
                rerank_applied = True

        results = []
        rank = 1
        for fid, sc in zip(cand_idx[:top_k], cand_scores[:top_k]):
            if int(fid) == -1:
                continue
            product_meta = self.mapper.get_by_faiss_id(int(fid))
            results.append({
                "rank": rank,
                "score": round(float(sc), 6),
                "product": product_meta,
            })
            rank += 1
        return results, rerank_applied
''', encoding="utf-8")

# ---------- app/main.py ----------
(APP_DIR / "main.py").write_text('''import os
from fastapi import FastAPI, File, UploadFile, Query, HTTPException, Request
from fastapi.responses import JSONResponse
from app.schemas import TextSearchRequest, SearchResponse, ProductMetadata, HealthResponse
from app.encoder import EnsembleCLIPEncoder
from app.index_service import FAISSIndexService
from app.exceptions import InvalidQueryError, InvalidImageError, IndexTrapError, VectorIDMappingError

DEFAULT_DATA_DIR = os.path.join(os.path.dirname(os.path.dirname(__file__)), "data")
INDEX_PATH   = os.getenv("FAISS_INDEX_PATH",   os.path.join(DEFAULT_DATA_DIR, "lookbench_gallery_hnsw_m16.faiss"))
CATALOG_PATH = os.getenv("PRODUCT_CATALOG_PATH", os.path.join(DEFAULT_DATA_DIR, "product_catalog.parquet"))


def create_app(encoder: EnsembleCLIPEncoder = None, index_service: FAISSIndexService = None) -> FastAPI:
    app = FastAPI(
        title="ABO Multimodal Visual Retrieval API (MURE v6 — Leakage-Free)",
        version="6.0.0",
        description="Multimodal visual search trên ABO với FashionCLIP ⊕ ViT-B/32 + Domain Centering + Inductive Dual-EKMA + MURE v6 Reranker"
    )

    app.state.encoder = encoder or EnsembleCLIPEncoder(data_dir=DEFAULT_DATA_DIR)
    app.state.index_service = index_service or FAISSIndexService(INDEX_PATH, CATALOG_PATH)

    @app.exception_handler(InvalidQueryError)
    async def _h1(_: Request, exc: InvalidQueryError):
        return JSONResponse(status_code=400, content={"detail": str(exc), "error_type": "InvalidQueryError"})

    @app.exception_handler(InvalidImageError)
    async def _h2(_: Request, exc: InvalidImageError):
        return JSONResponse(status_code=400, content={"detail": str(exc), "error_type": "InvalidImageError"})

    @app.exception_handler(IndexTrapError)
    async def _h3(_: Request, exc: IndexTrapError):
        return JSONResponse(status_code=503, content={"detail": str(exc), "error_type": "IndexTrapError"})

    @app.exception_handler(VectorIDMappingError)
    async def _h4(_: Request, exc: VectorIDMappingError):
        return JSONResponse(status_code=500, content={"detail": str(exc), "error_type": "VectorIDMappingError"})

    @app.get("/health", response_model=HealthResponse)
    async def health():
        idx: FAISSIndexService = app.state.index_service
        enc: EnsembleCLIPEncoder = app.state.encoder
        if idx is None or not idx.is_ready:
            raise IndexTrapError("FAISS index không khả dụng.")
        return HealthResponse(
            status="healthy",
            models=[m["tag"] for m in enc.models],
            index_loaded=idx.is_ready,
            transform_loaded=bool(getattr(enc, "transform_loaded", False)),
            total_indexed_vectors=int(idx.index.ntotal),
            total_catalog_products=len(idx.mapper),
            embedding_dim=enc.embedding_dim,
        )

    @app.post("/search/text", response_model=SearchResponse)
    async def search_text(payload: TextSearchRequest):
        enc: EnsembleCLIPEncoder = app.state.encoder
        idx: FAISSIndexService = app.state.index_service
        if idx is None or not idx.is_ready:
            raise IndexTrapError("FAISS index chưa sẵn sàng.")

        q_vec = enc.encode_text(payload.query)
        query_meta = {
            "category": payload.category or "",
            "main_attribute": payload.main_attribute or "",
            "other_attributes": payload.other_attributes or "",
        }
        hits, rr = idx.search(q_vec, top_k=payload.top_k, query_meta=query_meta, use_rerank=True)
        return SearchResponse(
            query_type="text", top_k=payload.top_k,
            total_results=len(hits), rerank_applied=rr, results=hits,
        )

    @app.post("/search/image", response_model=SearchResponse)
    async def search_image(
        file: UploadFile = File(..., description="JPEG/PNG/WEBP"),
        top_k: int = Query(default=10, ge=1, le=50),
        category: str = Query(default=""),
        main_attribute: str = Query(default=""),
        other_attributes: str = Query(default=""),
    ):
        enc: EnsembleCLIPEncoder = app.state.encoder
        idx: FAISSIndexService = app.state.index_service
        if idx is None or not idx.is_ready:
            raise IndexTrapError("FAISS index chưa sẵn sàng.")

        raw = await file.read()
        q_vec = enc.encode_image_bytes(raw, content_type=file.content_type)
        query_meta = {
            "category": category, "main_attribute": main_attribute,
            "other_attributes": other_attributes,
        }
        hits, rr = idx.search(q_vec, top_k=top_k, query_meta=query_meta, use_rerank=True)
        return SearchResponse(
            query_type="image", top_k=top_k,
            total_results=len(hits), rerank_applied=rr, results=hits,
        )

    @app.get("/products/{product_id}", response_model=ProductMetadata)
    async def get_product(product_id: str):
        idx: FAISSIndexService = app.state.index_service
        if idx is None or not idx.is_ready:
            raise IndexTrapError("Product catalog chưa sẵn sàng.")
        try:
            return ProductMetadata(**idx.mapper.get_by_product_id(product_id))
        except KeyError as exc:
            raise HTTPException(status_code=404, detail=str(exc)) from exc

    return app


app = create_app()
''', encoding="utf-8")

print("✅ Đã khởi tạo hoàn chỉnh FastAPI Backend (MURE v6 — ABO Edition) tại:", APP_DIR)


✅ Đã khởi tạo hoàn chỉnh FastAPI Backend (MURE v6 — ABO Edition) tại: /kaggle/working/fashion_search_backend/app


In [8]:
# ============ BỘ KIỂM THỬ TỰ ĐỘNG PYTEST (22 UNIT & API TESTS) & ĐÓNG GÓI ARTIFACTS ============
import subprocess

TESTS_DIR = pathlib.Path(BACKEND_ROOT) / "tests"
TESTS_DIR.mkdir(parents=True, exist_ok=True)

(TESTS_DIR / "test_backend.py").write_text('''import io
import os
import sys
import pytest
import numpy as np
from PIL import Image
from fastapi.testclient import TestClient

sys.path.insert(0, os.path.dirname(os.path.dirname(__file__)))

from app.encoder import EnsembleCLIPEncoder
from app.index_service import FAISSIndexService
from app.exceptions import InvalidQueryError, InvalidImageError, IndexTrapError, VectorIDMappingError
from app.main import create_app, INDEX_PATH, CATALOG_PATH, DEFAULT_DATA_DIR

EXPECTED_DIM = 1024


@pytest.fixture(scope="module")
def shared_encoder():
    return EnsembleCLIPEncoder(data_dir=DEFAULT_DATA_DIR)


@pytest.fixture(scope="module")
def shared_index_service():
    return FAISSIndexService(INDEX_PATH, CATALOG_PATH)


@pytest.fixture(scope="module")
def client(shared_encoder, shared_index_service):
    app = create_app(encoder=shared_encoder, index_service=shared_index_service)
    return TestClient(app)


def make_image_bytes(fmt="PNG", size=(224, 224), color=(120, 160, 200)) -> bytes:
    buf = io.BytesIO()
    Image.new("RGB", size, color).save(buf, format=fmt)
    return buf.getvalue()


def test_transform_weights_exported_and_loaded(shared_encoder):
    """Kiểm tra Fix 2: các file mean vector và ma trận W_QI, W_QT đã được lưu và nạp vào Encoder."""
    assert shared_encoder.transform_loaded is True
    assert shared_encoder.mean_img is not None and shared_encoder.mean_img.shape == (1, EXPECTED_DIM)
    assert shared_encoder.mean_txt is not None and shared_encoder.mean_txt.shape == (1, EXPECTED_DIM)
    assert shared_encoder.W_QI is not None and shared_encoder.W_QI.shape == (EXPECTED_DIM, EXPECTED_DIM)
    assert shared_encoder.W_QT is not None and shared_encoder.W_QT.shape == (EXPECTED_DIM, EXPECTED_DIM)


def test_index_and_metadata_strictly_synchronized(shared_index_service):
    """Kiểm tra Fix 4: số lượng vector trong FAISS index bằng chính xác số lượng sản phẩm trong Catalog."""
    assert shared_index_service.index.ntotal == len(shared_index_service.mapper)


def test_encoder_text_shape_and_norm(shared_encoder):
    emb = shared_encoder.encode_text("a modern beige linen sofa")
    assert emb.shape == (1, EXPECTED_DIM)
    assert emb.dtype == np.float32
    assert np.isclose(np.linalg.norm(emb), 1.0, atol=1e-5)


def test_encoder_text_rejects_empty(shared_encoder):
    with pytest.raises(InvalidQueryError):
        shared_encoder.encode_text("   ")


def test_encoder_image_shape_and_norm(shared_encoder):
    raw = make_image_bytes("JPEG")
    emb = shared_encoder.encode_image_bytes(raw, content_type="image/jpeg")
    assert emb.shape == (1, EXPECTED_DIM)
    assert np.isclose(np.linalg.norm(emb), 1.0, atol=1e-5)


def test_encoder_image_rejects_corrupted(shared_encoder):
    with pytest.raises(InvalidImageError):
        shared_encoder.encode_image_bytes(b"not_an_image", content_type="image/png")


def test_encoder_image_rejects_too_small(shared_encoder):
    tiny = make_image_bytes("PNG", size=(8, 8))
    with pytest.raises(InvalidImageError):
        shared_encoder.encode_image_bytes(tiny, content_type="image/png")


def test_mapping_consistency(shared_index_service):
    m0 = shared_index_service.mapper.get_by_faiss_id(0)
    assert m0["faiss_id"] == 0
    assert shared_index_service.mapper.get_by_product_id(m0["product_id"]) == m0
    assert "title" in m0 and "color" in m0 and "image_id" in m0


def test_mapping_out_of_bounds(shared_index_service):
    with pytest.raises(VectorIDMappingError):
        shared_index_service.mapper.get_by_faiss_id(-1)
    with pytest.raises(VectorIDMappingError):
        shared_index_service.mapper.get_by_faiss_id(999999)


def test_index_dimension_mismatch(shared_index_service):
    wrong = np.ones((1, 512), dtype=np.float32)
    with pytest.raises(IndexTrapError):
        shared_index_service.search(wrong, top_k=5)


def test_index_nan_vector(shared_index_service):
    nan = np.full((1, EXPECTED_DIM), np.nan, dtype=np.float32)
    with pytest.raises(IndexTrapError):
        shared_index_service.search(nan, top_k=5)


def test_rerank_boosts_matching_category(shared_index_service):
    vec = np.ascontiguousarray(
        shared_index_service.index.reconstruct(0).reshape(1, -1), dtype=np.float32
    )
    p0 = shared_index_service.mapper.get_by_faiss_id(0)

    hits_no, rr_no = shared_index_service.search(vec, top_k=10, use_rerank=False)
    hits_yes, rr_yes = shared_index_service.search(
        vec, top_k=10,
        query_meta={
            "category": p0["category"],
            "main_attribute": p0["main_attribute"],
            "other_attributes": p0["other_attributes"],
        },
        use_rerank=True,
    )
    assert rr_no is False
    assert rr_yes is True
    assert hits_no[0]["product"]["faiss_id"] == 0
    assert hits_yes[0]["product"]["faiss_id"] == 0


def test_health(client):
    r = client.get("/health")
    assert r.status_code == 200
    d = r.json()
    assert d["status"] == "healthy"
    assert d["index_loaded"] is True
    assert d["transform_loaded"] is True
    assert d["embedding_dim"] == EXPECTED_DIM
    assert d["total_indexed_vectors"] == d["total_catalog_products"]
    assert len(d["models"]) == 2


def test_search_text_no_meta(client):
    r = client.post("/search/text", json={"query": "a modern beige linen sofa", "top_k": 5})
    assert r.status_code == 200
    b = r.json()
    assert b["query_type"] == "text"
    assert b["total_results"] == 5
    assert b["rerank_applied"] is False


def test_search_text_with_meta(client):
    r = client.post("/search/text", json={
        "query": "a modern beige linen sofa",
        "top_k": 5,
        "category": "sofa",
        "main_attribute": "beige",
        "other_attributes": "linen, modern",
    })
    assert r.status_code == 200
    assert r.json()["rerank_applied"] is True


def test_search_text_empty_rejected(client):
    r = client.post("/search/text", json={"query": "   ", "top_k": 5})
    assert r.status_code == 422


def test_search_text_invalid_top_k(client):
    assert client.post("/search/text", json={"query": "abc", "top_k": 0}).status_code == 422
    assert client.post("/search/text", json={"query": "abc", "top_k": 999}).status_code == 422


def test_search_image_success(client):
    img = make_image_bytes("PNG")
    r = client.post("/search/image?top_k=4", files={"file": ("sample.png", img, "image/png")})
    assert r.status_code == 200
    b = r.json()
    assert b["query_type"] == "image"
    assert len(b["results"]) == 4


def test_search_image_with_meta(client):
    img = make_image_bytes("JPEG")
    r = client.post(
        "/search/image?top_k=5&category=chair&main_attribute=black",
        files={"file": ("sample.jpg", img, "image/jpeg")}
    )
    assert r.status_code == 200
    assert r.json()["rerank_applied"] is True


def test_search_image_invalid_upload(client):
    r = client.post("/search/image?top_k=5", files={"file": ("mal.txt", b"hello", "text/plain")})
    assert r.status_code == 400
    assert r.json()["error_type"] == "InvalidImageError"


def test_get_product_ok_and_404(client, shared_index_service):
    pid = shared_index_service.mapper.get_by_faiss_id(0)["product_id"]
    assert client.get(f"/products/{pid}").status_code == 200
    assert client.get("/products/nope_999").status_code == 404


def test_endpoint_index_unavailable(shared_encoder, shared_index_service):
    app = create_app(encoder=shared_encoder, index_service=shared_index_service)
    app.state.index_service = None
    broken = TestClient(app)
    r = broken.post("/search/text", json={"query": "wooden table", "top_k": 5})
    assert r.status_code == 503
    assert r.json()["error_type"] == "IndexTrapError"
''', encoding="utf-8")

# ---------- requirements.txt & Dockerfile ----------
pathlib.Path(BACKEND_ROOT, "requirements.txt").write_text(
    "fastapi>=0.110.0\nuvicorn[standard]>=0.29.0\npython-multipart>=0.0.9\n"
    "pydantic>=2.6.0\nfaiss-cpu>=1.8.0\nopen_clip_torch>=2.24.0\n"
    "torch>=2.1.0\npillow>=10.0.0\npandas>=2.0.0\npyarrow>=15.0.0\n"
    "numpy>=1.26.0\npytest>=8.0.0\nhttpx>=0.27.0\n",
    encoding="utf-8"
)

pathlib.Path(BACKEND_ROOT, "Dockerfile").write_text(
    "FROM python:3.11-slim\n"
    "WORKDIR /app\n"
    "COPY requirements.txt .\n"
    "RUN pip install --no-cache-dir -r requirements.txt\n"
    "COPY . .\n"
    "EXPOSE 8000\n"
    'CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]\n',
    encoding="utf-8"
)

print("🧪 Chạy bộ kiểm thử tự động pytest (22 tests bao gồm kiểm tra Transform Weights & Index-Metadata Sync)...")
res = subprocess.run(
    ["pytest", "-v", os.path.join(BACKEND_ROOT, "tests", "test_backend.py")],
    capture_output=True, text=True
)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr)
    raise RuntimeError("❌ Kiểm thử thất bại!")

with open(os.path.join(REPORT_DIR, "pytest_unified_log.txt"), "w", encoding="utf-8") as f:
    f.write(res.stdout)

# ---------- ĐÓNG GÓI TOÀN BỘ ARTIFACTS DAY 3 + DAY 4 ----------
MANIFEST = {
    "created_at": datetime.now().isoformat(),
    "pipeline": "Unified Day 3 + Day 4 — MURE v6 Inductive Dual-EKMA on Authentic Amazon Berkeley Objects (ABO)",
    "models": ["FashionCLIP (0.7)", "ViT-B/32-laion (0.3)", "Inductive_Dual_EKMA_1024"],
    "best_config": BEST_CFG,
}
with open(os.path.join(ART_DIR, "MANIFEST_DAY3_DAY4.json"), "w", encoding="utf-8") as f:
    json.dump(MANIFEST, f, indent=2, ensure_ascii=False)

OUTPUT_ZIP = "/kaggle/working/day3_day4_unified_final.zip"
with zipfile.ZipFile(OUTPUT_ZIP, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for folder in [BACKEND_ROOT, ART_DIR, REPORT_DIR]:
        for root, _, files in os.walk(folder):
            for file in files:
                full = os.path.join(root, file)
                rel = os.path.relpath(full, "/kaggle/working")
                zf.write(full, arcname=rel)

with zipfile.ZipFile(OUTPUT_ZIP) as zf:
    assert zf.testzip() is None

size_mb = os.path.getsize(OUTPUT_ZIP) / 1e6
print(f"🎉 HOÀN TẤT UNIFIED DAY 3 + DAY 4 TRÊN ABO (MURE v6 — LEAKAGE-FREE)! → {OUTPUT_ZIP} ({size_mb:.2f} MB)")


🧪 Chạy bộ kiểm thử tự động pytest (22 tests bao gồm kiểm tra Transform Weights & Index-Metadata Sync)...
============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /kaggle/working
plugins: typeguard-4.6.0, anyio-4.14.2, langsmith-0.12.1
collecting ... collected 22 items

fashion_search_backend/tests/test_backend.py::test_transform_weights_exported_and_loaded PASSED [  4%]
fashion_search_backend/tests/test_backend.py::test_index_and_metadata_strictly_synchronized PASSED [  9%]
fashion_search_backend/tests/test_backend.py::test_encoder_text_shape_and_norm PASSED [ 13%]
fashion_search_backend/tests/test_backend.py::test_encoder_text_rejects_empty PASSED [ 18%]
fashion_search_backend/tests/test_backend.py::test_encoder_image_shape_and_norm PASSED [ 22%]
fashion_search_backend/tests/test_backend.py::test_encoder_image_rejects_corrupted PASSED [ 27%]
fashion